In [4]:
# ============================================================
# STEP 15 — CELL 1
# INITIALIZE AGENT ENVIRONMENT
# ============================================================

import os
import sys
import json
import pickle
import warnings
import importlib.metadata

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")


print("=" * 80)
print("STEP 15 — E-COMMERCE AI AGENT")
print("=" * 80)


# ------------------------------------------------------------
# 1. Runtime information
# ------------------------------------------------------------

print("\n[1] RUNTIME")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


for package in [
    "transformers",
    "sentence-transformers",
    "faiss-cpu",
    "implicit",
    "bitsandbytes",
]:

    try:
        version = importlib.metadata.version(package)

    except importlib.metadata.PackageNotFoundError:
        version = "NOT INSTALLED"

    print(f"{package}: {version}")


# ------------------------------------------------------------
# 2. Output directory
# ------------------------------------------------------------

STEP15_DIR = "/kaggle/working/step15"

os.makedirs(
    STEP15_DIR,
    exist_ok=True
)

print("\n[2] OUTPUT DIRECTORY")
print(STEP15_DIR)


# ------------------------------------------------------------
# 3. Discover input artifacts
# ------------------------------------------------------------

INPUT_ROOT = "/kaggle/input"

REQUIRED_FILES = [
    "unified_products.parquet",
    "unified_reviews.parquet",
    "product_faiss_moi.index",
    "product_embedding_metadata.parquet",
    "step11_config.json",
    "bpr_model.pkl",
    "cf_mappings.pkl",
    "step12_config.json",
    "step13_config.json",
    "step14_config.json",
    "step14_final_test.json",
    "step14_ke_evaluation.json",
    "knowledge_edits.json",
]


def discover_artifacts(root, required_files):

    matches = {
        filename: []
        for filename in required_files
    }

    for current_root, _, filenames in os.walk(root):

        for filename in filenames:

            if filename in matches:

                matches[filename].append(
                    os.path.join(
                        current_root,
                        filename
                    )
                )

    return matches


artifact_matches = discover_artifacts(
    INPUT_ROOT,
    REQUIRED_FILES
)


print("\n[3] INPUT ARTIFACTS")
print("-" * 80)

artifact_paths = {}

missing_files = []
duplicate_files = []


for filename in REQUIRED_FILES:

    paths = artifact_matches[filename]

    if len(paths) == 0:

        print(f"[MISSING] {filename}")

        missing_files.append(filename)

    elif len(paths) == 1:

        artifact_paths[filename] = paths[0]

        print(f"[OK] {filename}")
        print(f"     {paths[0]}")

    else:

        print(f"[MULTIPLE] {filename}")

        for path in paths:
            print(f"     {path}")

        duplicate_files.append(filename)


# ------------------------------------------------------------
# 4. Save discovery report
# ------------------------------------------------------------

discovery_report = {

    "step": "step15_agent_tool_calling",

    "input_root": INPUT_ROOT,

    "output_dir": STEP15_DIR,

    "artifact_matches": artifact_matches,

    "missing_files": missing_files,

    "duplicate_files": duplicate_files,

    "cuda_available": torch.cuda.is_available(),
}


report_path = os.path.join(
    STEP15_DIR,
    "step15_artifact_discovery.json"
)


with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        discovery_report,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

print(
    "Required artifacts:",
    len(REQUIRED_FILES)
)

print(
    "Unique artifacts found:",
    len(artifact_paths)
)

print(
    "Missing:",
    len(missing_files)
)

print(
    "Multiple matches:",
    len(duplicate_files)
)


if missing_files:

    print("\nACTION REQUIRED:")
    print("Add the Kaggle Dataset containing the missing files.")

elif duplicate_files:

    print("\nACTION REQUIRED:")
    print(
        "Select the correct dataset versions "
        "before loading models."
    )

else:

    print("\nPASS: STEP 15 INPUT ARTIFACTS READY")


print("\nDiscovery report:", report_path)

STEP 15 — E-COMMERCE AI AGENT

[1] RUNTIME
Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
transformers: 5.0.0
sentence-transformers: 5.4.1
faiss-cpu: 1.15.1
implicit: 0.7.3
bitsandbytes: 0.50.2

[2] OUTPUT DIRECTORY
/kaggle/working/step15

[3] INPUT ARTIFACTS
--------------------------------------------------------------------------------
[OK] unified_products.parquet
     /kaggle/input/datasets/trnquangtriu/15dataset/unified_products.parquet
[OK] unified_reviews.parquet
     /kaggle/input/datasets/trnquangtriu/15dataset/unified_reviews.parquet
[OK] product_faiss_moi.index
     /kaggle/input/datasets/trnquangtriu/15dataset/product_faiss_moi.index
[OK] product_embedding_metadata.parquet
     /kaggle/input/datasets/trnquangtriu/15dataset/product_embedding_metadata.parquet
[OK] step11_config.json
     /kaggle/input/datasets/trnquangtriu/15dataset/step11_config.json
[OK] bpr_model.pkl
     /kaggle/input/datasets/trnquangtriu/15dataset/bpr_model.pkl
[OK] cf_mapping

In [2]:
# ============================================================
# STEP 15 — CELL 2
# INSTALL REQUIRED DEPENDENCIES
# ============================================================

%pip install -q "faiss-cpu>=1.9,<2" "implicit>=0.7,<1" "bitsandbytes>=0.46.1,<1"

print("=" * 80)
print("DEPENDENCY INSTALLATION FINISHED")
print("=" * 80)
print("Restart the Kaggle session before running Cell 3.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 94.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.9 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.
DEPENDENCY INSTALLATION FINISHED
Restart the Kaggle session before running Cell 3.


In [5]:
# ============================================================
# STEP 15 — CELL 3
# LOAD DATA, FAISS, BPR AND KNOWLEDGE EDITS
# ============================================================

import os
import json
import pickle

import numpy as np
import pandas as pd
import faiss
import implicit
import bitsandbytes
import torch

print("=" * 80)
print("STEP 15 — LOAD BACKEND ARTIFACTS")
print("=" * 80)


# ------------------------------------------------------------
# 1. Load product and review datasets
# ------------------------------------------------------------

print("\n[1] LOADING DATASETS")

products = pd.read_parquet(
    artifact_paths["unified_products.parquet"]
)

reviews = pd.read_parquet(
    artifact_paths["unified_reviews.parquet"]
)

product_metadata = pd.read_parquet(
    artifact_paths["product_embedding_metadata.parquet"]
)

print("Products:", products.shape)
print("Reviews:", reviews.shape)
print("FAISS metadata:", product_metadata.shape)


# ------------------------------------------------------------
# 2. Load FAISS index
# ------------------------------------------------------------

print("\n[2] LOADING FAISS")

faiss_index = faiss.read_index(
    artifact_paths["product_faiss_moi.index"]
)

print("Index type:", type(faiss_index).__name__)
print("Vectors:", faiss_index.ntotal)
print("Dimension:", faiss_index.d)


# ------------------------------------------------------------
# 3. Load BPR model
# ------------------------------------------------------------

print("\n[3] LOADING BPR")

with open(
    artifact_paths["bpr_model.pkl"],
    "rb"
) as f:
    bpr_model = pickle.load(f)

with open(
    artifact_paths["cf_mappings.pkl"],
    "rb"
) as f:
    cf_mappings = pickle.load(f)

print("BPR type:", type(bpr_model).__name__)
print("User factors:", bpr_model.user_factors.shape)
print("Item factors:", bpr_model.item_factors.shape)
print("Mapping keys:", list(cf_mappings.keys()))


# ------------------------------------------------------------
# 4. Load Step 14 artifacts
# ------------------------------------------------------------

print("\n[4] LOADING KNOWLEDGE EDITING ARTIFACTS")


def load_json_artifact(filename):

    with open(
        artifact_paths[filename],
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


step11_config = load_json_artifact(
    "step11_config.json"
)

step12_config = load_json_artifact(
    "step12_config.json"
)

step13_config = load_json_artifact(
    "step13_config.json"
)

step14_config = load_json_artifact(
    "step14_config.json"
)

step14_final_test = load_json_artifact(
    "step14_final_test.json"
)

step14_ke_evaluation = load_json_artifact(
    "step14_ke_evaluation.json"
)

knowledge_edits = load_json_artifact(
    "knowledge_edits.json"
)

print(
    "KE method:",
    step14_config["knowledge_editing"]["method"]
)

print(
    "Parameter editing:",
    step14_config["knowledge_editing"]["parameter_weight_editing"]
)

print(
    "Knowledge edit store type:",
    type(knowledge_edits).__name__
)

if isinstance(knowledge_edits, dict):
    print(
        "Knowledge edit store keys:",
        list(knowledge_edits.keys())
    )
elif isinstance(knowledge_edits, list):
    print(
        "Knowledge edit records:",
        len(knowledge_edits)
    )


# ------------------------------------------------------------
# 5. Validate FAISS mapping
# ------------------------------------------------------------

print("\n[5] VALIDATION")

assert len(products) == 164926, (
    "Unexpected product count"
)

assert len(reviews) == 767179, (
    "Unexpected review count"
)

assert faiss_index.ntotal == len(
    product_metadata
), "FAISS/metadata count mismatch"

assert faiss_index.d == 384, (
    "Unexpected embedding dimension"
)

assert products["product_id"].is_unique, (
    "Duplicate product_id"
)

assert reviews["review_id"].is_unique, (
    "Duplicate review_id"
)

assert product_metadata["faiss_row_id"].is_unique, (
    "Duplicate faiss_row_id"
)

assert set(
    product_metadata["faiss_row_id"].astype(int)
) == set(
    range(faiss_index.ntotal)
), "FAISS row IDs are not contiguous"


required_mapping_keys = {
    "user_to_idx",
    "item_to_idx",
    "idx_to_user",
    "idx_to_item"
}

assert required_mapping_keys.issubset(
    cf_mappings.keys()
), "Missing CF mapping keys"

assert (
    bpr_model.user_factors.shape[0]
    == len(cf_mappings["user_to_idx"])
), "BPR user mapping mismatch"

assert (
    bpr_model.item_factors.shape[0]
    == len(cf_mappings["item_to_idx"])
), "BPR item mapping mismatch"


# ------------------------------------------------------------
# 6. Prepare lookup structures
# ------------------------------------------------------------

products_by_id = products.set_index(
    "product_id",
    drop=False
)

metadata_by_faiss_row = (
    product_metadata.set_index(
        "faiss_row_id",
        drop=False
    )
)

user_to_idx = cf_mappings["user_to_idx"]
item_to_idx = cf_mappings["item_to_idx"]

print("Products lookup:", len(products_by_id))
print("FAISS lookup:", len(metadata_by_faiss_row))
print("Known BPR users:", len(user_to_idx))
print("Known BPR items:", len(item_to_idx))


print("\n" + "=" * 80)
print("BACKEND ARTIFACT SUMMARY")
print("=" * 80)

print("Products:", len(products))
print("Reviews:", len(reviews))
print("FAISS vectors:", faiss_index.ntotal)
print("Embedding dimension:", faiss_index.d)
print("BPR users:", bpr_model.user_factors.shape[0])
print("BPR items:", bpr_model.item_factors.shape[0])
print("Knowledge Editing:", step14_config["knowledge_editing"]["method"])

print("\nPASS: STEP 15 BACKEND ARTIFACTS LOADED")

STEP 15 — LOAD BACKEND ARTIFACTS

[1] LOADING DATASETS
Products: (164926, 60)
Reviews: (767179, 16)
FAISS metadata: (164926, 17)

[2] LOADING FAISS
Index type: IndexFlatIP
Vectors: 164926
Dimension: 384

[3] LOADING BPR
BPR type: BayesianPersonalizedRanking
User factors: (26903, 65)
Item factors: (15546, 65)
Mapping keys: ['user_to_idx', 'item_to_idx', 'idx_to_user', 'idx_to_item']

[4] LOADING KNOWLEDGE EDITING ARTIFACTS
KE method: retrieval_time_knowledge_edit_overlay
Parameter editing: False
Knowledge edit store type: list
Knowledge edit records: 3

[5] VALIDATION
Products lookup: 164926
FAISS lookup: 164926
Known BPR users: 26903
Known BPR items: 15546

BACKEND ARTIFACT SUMMARY
Products: 164926
Reviews: 767179
FAISS vectors: 164926
Embedding dimension: 384
BPR users: 26903
BPR items: 15546
Knowledge Editing: retrieval_time_knowledge_edit_overlay

PASS: STEP 15 BACKEND ARTIFACTS LOADED


In [6]:
# ============================================================
# STEP 15 — CELL 4
# LOAD EMBEDDING MODEL + VALIDATE SEARCH SCHEMA
# ============================================================

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer


print("=" * 80)
print("STEP 15 — EMBEDDING MODEL")
print("=" * 80)


# ------------------------------------------------------------
# 1. Load original embedding model
# ------------------------------------------------------------

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("\n[1] LOADING MODEL")
print("Model:", EMBEDDING_MODEL_NAME)


product_query_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda"
    if torch.cuda.is_available()
    else "cpu"
)

product_query_model.max_seq_length = 128

print(
    "Embedding dimension:",
    product_query_model.get_sentence_embedding_dimension()
)

print(
    "Max sequence length:",
    product_query_model.max_seq_length
)


# ------------------------------------------------------------
# 2. Validate embedding dimensions
# ------------------------------------------------------------

assert (
    product_query_model.get_sentence_embedding_dimension()
    == faiss_index.d
), "Embedding dimension mismatch"


# ------------------------------------------------------------
# 3. Inspect required product fields
# ------------------------------------------------------------

print("\n[2] PRODUCT SCHEMA")

required_product_columns = [
    "product_id",
    "product_name",
    "source",
    "price",
    "currency",
    "average_rating",
    "review_count",
    "description",
    "product_features",
    "product_url",
]

for column in required_product_columns:

    exists = column in products.columns

    print(
        f"{column:<25}",
        "OK" if exists else "MISSING"
    )

    assert exists, (
        f"Missing product column: {column}"
    )


# ------------------------------------------------------------
# 4. Inspect FAISS metadata fields
# ------------------------------------------------------------

print("\n[3] FAISS METADATA SCHEMA")

print(
    product_metadata.columns.tolist()
)


required_metadata_columns = [
    "faiss_row_id",
    "product_id"
]

for column in required_metadata_columns:

    assert column in product_metadata.columns, (
        f"Missing metadata column: {column}"
    )


# ------------------------------------------------------------
# 5. Check product mapping integrity
# ------------------------------------------------------------

metadata_product_ids = set(
    product_metadata["product_id"].astype(str)
)

dataset_product_ids = set(
    products["product_id"].astype(str)
)

missing_product_ids = (
    metadata_product_ids
    - dataset_product_ids
)

assert not missing_product_ids, (
    "Some FAISS products are missing "
    "from the product dataset"
)


print("\n[4] MAPPING VALIDATION")

print(
    "FAISS products:",
    len(metadata_product_ids)
)

print(
    "Dataset products:",
    len(dataset_product_ids)
)

print(
    "Missing products:",
    len(missing_product_ids)
)


# ------------------------------------------------------------
# 6. Check Knowledge Edit Store
# ------------------------------------------------------------

print("\n[5] KNOWLEDGE EDIT STORE")

print(
    "Total edit records:",
    len(knowledge_edits)
)

print(
    "Record keys:"
)

if knowledge_edits:

    print(
        list(knowledge_edits[0].keys())
    )


print(
    "\nEdit statuses:"
)

for edit in knowledge_edits:

    print(
        "ID:",
        edit.get("edit_id"),

        "| Product:",
        edit.get("product_id"),

        "| Field:",
        edit.get("field"),

        "| Status:",
        edit.get("status")
    )


print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

print(
    "Embedding dimension matches FAISS:",
    True
)

print(
    "All FAISS product IDs exist:",
    True
)

print(
    "\nPASS: SEARCH MODEL AND SCHEMA READY"
)

STEP 15 — EMBEDDING MODEL

[1] LOADING MODEL
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
Max sequence length: 128

[2] PRODUCT SCHEMA
product_id                OK
product_name              OK
source                    OK
price                     OK
currency                  OK
average_rating            OK
review_count              OK
description               OK
product_features          OK
product_url               OK

[3] FAISS METADATA SCHEMA
['faiss_row_id', 'product_id', 'product_name', 'source', 'category', 'subcategory', 'category_path', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'description', 'product_features', 'product_url', 'retrieval_text', 'embedding_text_v2']

[4] MAPPING VALIDATION
FAISS products: 164926
Dataset products: 164926
Missing products: 0

[5] KNOWLEDGE EDIT STORE
Total edit records: 3
Record keys:
['edit_id', 'product_id', 'field', 'old_value', 'new_value', 'reason', 'evidence', 'provenance', 'created_at', 'status', 'version', 'status_updated_at']

Edit statuses:
ID: 00f330af-10b3-4f70-9eeb-a0c815e60

In [7]:
# ============================================================
# STEP 15 — CELL 5
# KE-AWARE PRODUCT SEARCH TOOL
# ============================================================

import re
import math
import numpy as np
import pandas as pd

from collections import defaultdict


print("=" * 80)
print("STEP 15 — PRODUCT SEARCH TOOL")
print("=" * 80)


# ------------------------------------------------------------
# 1. Normalize values
# ------------------------------------------------------------

def clean_value(value):

    if value is None:
        return ""

    if isinstance(value, (list, tuple)):
        return " ".join(
            clean_value(x)
            for x in value
        )

    if isinstance(value, np.ndarray):
        return " ".join(
            clean_value(x)
            for x in value.tolist()
        )

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value).strip()


def safe_float(value, default=None):

    try:
        result = float(value)

        if not np.isfinite(result):
            return default

        return result

    except (TypeError, ValueError):
        return default


# ------------------------------------------------------------
# 2. Read active Knowledge Edits
# ------------------------------------------------------------

EDITABLE_FIELDS = {
    "product_name",
    "description",
    "product_features",
    "price",
    "currency",
    "product_url"
}


def get_active_product_edits(product_id):

    active = [
        edit
        for edit in knowledge_edits

        if (
            str(edit.get("product_id"))
            == str(product_id)
        )

        and edit.get("status") == "active"

        and edit.get("field") in EDITABLE_FIELDS
    ]

    return sorted(
        active,
        key=lambda e: (
            int(e.get("version", 0)),
            str(e.get("created_at", "")),
            str(e.get("edit_id", ""))
        )
    )


def get_effective_product(product_id):

    product_id = str(product_id)

    if product_id not in products_by_id.index:
        return None

    row = products_by_id.loc[
        product_id
    ].to_dict()

    applied_edits = []

    for edit in get_active_product_edits(
        product_id
    ):

        field = edit["field"]

        row[field] = edit["new_value"]

        applied_edits.append({
            "edit_id": edit["edit_id"],
            "field": field,
            "version": edit.get("version")
        })

    row["knowledge_edit_applied"] = (
        len(applied_edits) > 0
    )

    row["applied_edits"] = applied_edits

    return row


# ------------------------------------------------------------
# 3. Build KE embedding overlay
# ------------------------------------------------------------

def build_product_embedding_text(product):

    parts = [
        clean_value(
            product.get("product_name")
        ),
        clean_value(
            product.get("description")
        ),
        clean_value(
            product.get("product_features")
        )
    ]

    return " ".join(
        part
        for part in parts
        if part
    )


def build_active_ke_overlay():

    active_product_ids = sorted({
        str(edit["product_id"])

        for edit in knowledge_edits

        if (
            edit.get("status") == "active"
            and edit.get("field") in EDITABLE_FIELDS
        )
    })

    overlay = {}

    for product_id in active_product_ids:

        product = get_effective_product(
            product_id
        )

        if product is None:
            continue

        text = build_product_embedding_text(
            product
        )

        if not text:
            continue

        vector = product_query_model.encode(
            [text],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )[0].astype("float32")

        overlay[product_id] = vector

    return overlay


ke_vector_overlay = build_active_ke_overlay()


print(
    "Active KE vector overlay:",
    len(ke_vector_overlay)
)


# ------------------------------------------------------------
# 4. Query requirements
# ------------------------------------------------------------

def parse_search_requirements(query):

    query_lower = query.lower()

    requirements = {
        "capacity_gb": None,
        "max_price": None,
        "currency": None
    }

    # Capacity: 64GB / 128 GB / 1TB
    capacity_match = re.search(
        r"\b(\d+(?:\.\d+)?)\s*(gb|tb)\b",
        query_lower
    )

    if capacity_match:

        value = float(
            capacity_match.group(1)
        )

        unit = capacity_match.group(2)

        if unit == "tb":
            value *= 1024

        requirements["capacity_gb"] = value

    # Budget and currency
    budget_patterns = [
        r"(?:under|below|less than|max|tối đa|dưới|không quá)\s*\$?\s*(\d+(?:\.\d+)?)\s*(usd|vnd|đ|đồng)?",
        r"(?:under|below|less than|max|tối đa|dưới|không quá)\s*(\d+(?:\.\d+)?)\s*(usd|vnd|đ|đồng)",
    ]

    for pattern in budget_patterns:

        match = re.search(
            pattern,
            query_lower
        )

        if match:

            requirements["max_price"] = float(
                match.group(1)
            )

            currency = match.group(2)

            if currency:
                requirements["currency"] = (
                    "VND"
                    if currency in {"vnd", "đ", "đồng"}
                    else "USD"
                )

            elif "$" in match.group(0):
                requirements["currency"] = "USD"

            break

    return requirements


def extract_title_capacities_gb(title):

    title = clean_value(title).lower()

    matches = re.findall(
        r"\b(\d+(?:\.\d+)?)\s*(gb|tb)\b",
        title
    )

    capacities = []

    for number, unit in matches:

        value = float(number)

        if unit == "tb":
            value *= 1024

        capacities.append(value)

    return capacities


# ------------------------------------------------------------
# 5. Validate hard constraints
# ------------------------------------------------------------

def product_matches_requirements(
    product,
    requirements
):

    requested_capacity = requirements[
        "capacity_gb"
    ]

    if requested_capacity is not None:

        capacities = extract_title_capacities_gb(
            product.get("product_name")
        )

        if not any(
            math.isclose(
                capacity,
                requested_capacity,
                rel_tol=0,
                abs_tol=0.001
            )

            for capacity in capacities
        ):
            return False

    max_price = requirements[
        "max_price"
    ]

    if max_price is not None:

        price = safe_float(
            product.get("price")
        )

        if price is None:
            return False

        if price > max_price:
            return False

        required_currency = requirements[
            "currency"
        ]

        if required_currency is not None:

            actual_currency = clean_value(
                product.get("currency")
            ).upper()

            if actual_currency != required_currency:
                return False

    return True


# ------------------------------------------------------------
# 6. Semantic ranking
# ------------------------------------------------------------

def calculate_semantic_rank_score(
    semantic_score,
    rating,
    review_count
):

    rating_value = safe_float(
        rating,
        default=0.0
    )

    review_value = safe_float(
        review_count,
        default=0.0
    )

    rating_norm = np.clip(
        rating_value / 5.0,
        0,
        1
    )

    review_norm = np.clip(
        np.log1p(
            max(review_value, 0)
        ) / np.log1p(100000),
        0,
        1
    )

    return float(
        0.75 * semantic_score
        + 0.20 * rating_norm
        + 0.05 * review_norm
    )


# ------------------------------------------------------------
# 7. Search tool
# ------------------------------------------------------------

def search_products(
    query,
    top_k=5,
    search_k=1000
):

    if not isinstance(query, str):
        raise TypeError(
            "query must be a string"
        )

    if not query.strip():
        raise ValueError(
            "query cannot be empty"
        )

    if top_k < 1 or search_k < 1:
        raise ValueError(
            "top_k and search_k must be positive"
        )

    # Rebuild overlay so newly activated edits
    # are reflected in subsequent searches.
    overlay = build_active_ke_overlay()

    requirements = parse_search_requirements(
        query
    )

    query_vector = product_query_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_vector,
        min(search_k, faiss_index.ntotal)
    )

    # Base FAISS candidates
    candidate_scores = {}

    for row_id, score in zip(
        indices[0],
        distances[0]
    ):

        if row_id < 0:
            continue

        metadata = metadata_by_faiss_row.loc[
            int(row_id)
        ]

        product_id = str(
            metadata["product_id"]
        )

        # Do not use stale base embedding score
        # for actively edited products.
        if product_id in overlay:
            continue

        candidate_scores[product_id] = {
            "semantic_score": float(score),
            "retrieval_source": "base_faiss"
        }

    # Edited product scores replace base scores.
    for product_id, vector in overlay.items():

        score = float(
            np.dot(
                query_vector[0],
                vector
            )
        )

        candidate_scores[product_id] = {
            "semantic_score": score,
            "retrieval_source": "ke_overlay"
        }

    eligible = []

    for product_id, retrieval in (
        candidate_scores.items()
    ):

        product = get_effective_product(
            product_id
        )

        if product is None:
            continue

        if not product_matches_requirements(
            product,
            requirements
        ):
            continue

        semantic_score = retrieval[
            "semantic_score"
        ]

        rank_score = (
            calculate_semantic_rank_score(
                semantic_score,
                product.get("average_rating"),
                product.get("review_count")
            )
        )

        eligible.append({

            "product_id": product_id,

            "title": clean_value(
                product.get("product_name")
            ),

            "source": clean_value(
                product.get("source")
            ),

            "price": safe_float(
                product.get("price")
            ),

            "currency": clean_value(
                product.get("currency")
            ),

            "rating": safe_float(
                product.get("average_rating")
            ),

            "review_count": safe_float(
                product.get("review_count"),
                default=0
            ),

            "semantic_score": semantic_score,

            "final_score": rank_score,

            "retrieval_source": retrieval[
                "retrieval_source"
            ],

            "knowledge_edit_applied": product[
                "knowledge_edit_applied"
            ],

            "applied_edits": product[
                "applied_edits"
            ]
        })

    eligible.sort(
        key=lambda x: (
            -x["final_score"],
            x["product_id"]
        )
    )

    return {

        "tool": "search_products",

        "query": query,

        "requirements": requirements,

        "searched_candidates": len(
            candidate_scores
        ),

        "eligible_candidates": len(
            eligible
        ),

        "active_ke_products": len(
            overlay
        ),

        "products": eligible[:top_k]
    }


print(
    "\nPASS: search_products TOOL DEFINED"
)

STEP 15 — PRODUCT SEARCH TOOL
Active KE vector overlay: 0

PASS: search_products TOOL DEFINED


In [8]:
# ============================================================
# STEP 15 — CELL 6
# TEST PRODUCT SEARCH TOOL
# ============================================================

test_query = (
    "microSD 64GB under 20 USD "
    "for photos and video"
)

search_result = search_products(
    query=test_query,
    top_k=5,
    search_k=1000
)


print("=" * 80)
print("PRODUCT SEARCH TEST")
print("=" * 80)

print(
    "Query:",
    search_result["query"]
)

print(
    "Requirements:",
    search_result["requirements"]
)

print(
    "Searched candidates:",
    search_result["searched_candidates"]
)

print(
    "Eligible candidates:",
    search_result["eligible_candidates"]
)

print(
    "Active KE products:",
    search_result["active_ke_products"]
)


print("\nTOP 5 PRODUCTS")
print("-" * 80)

for rank, product in enumerate(
    search_result["products"],
    start=1
):

    print(
        f"\n#{rank}: {product['title']}"
    )

    print(
        "Product ID:",
        product["product_id"]
    )

    print(
        "Price:",
        product["price"],
        product["currency"]
    )

    print(
        "Rating:",
        product["rating"]
    )

    print(
        "Semantic score:",
        round(
            product["semantic_score"],
            4
        )
    )

    print(
        "Final score:",
        round(
            product["final_score"],
            4
        )
    )

    print(
        "Retrieval source:",
        product["retrieval_source"]
    )

    print(
        "KE applied:",
        product[
            "knowledge_edit_applied"
        ]
    )


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert search_result["requirements"][
    "capacity_gb"
] == 64

assert search_result["requirements"][
    "max_price"
] == 20

assert search_result["requirements"][
    "currency"
] == "USD"

assert search_result[
    "active_ke_products"
] == 0

assert len(
    search_result["products"]
) == 5


for product in search_result["products"]:

    assert product["price"] <= 20

    assert product["currency"] == "USD"

    assert not product[
        "knowledge_edit_applied"
    ]


print(
    "\nPASS: PRODUCT SEARCH TOOL"
)

PRODUCT SEARCH TEST
Query: microSD 64GB under 20 USD for photos and video
Requirements: {'capacity_gb': 64.0, 'max_price': 20.0, 'currency': 'USD'}
Searched candidates: 1000
Eligible candidates: 80
Active KE products: 0

TOP 5 PRODUCTS
--------------------------------------------------------------------------------

#1: SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
Product ID: amazon_B0BRZPCQNJ
Price: 16.99 USD
Rating: 4.7
Semantic score: 0.7204
Final score: 0.7783
Retrieval source: base_faiss
KE applied: False

#2: SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
Product ID: amazon_B07XDCZ9J3
Price: 12.78 USD
Rating: 4.7
Semantic score: 0.6965
Final score: 0.7504
Retrieval source: base_faiss
KE applied: False

#3: Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
Product ID: amazon_B0C1H1Q3C1

In [9]:
# ============================================================
# STEP 15 — CELL 7
# PRODUCT TYPE VALIDATION + PACK INFORMATION
# ============================================================

import re
import math


print("=" * 80)
print("STEP 15 — PRODUCT TYPE VALIDATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. Detect requested card type
# ------------------------------------------------------------

def detect_requested_card_type(query):

    text = str(query).lower()

    if re.search(
        r"\bmicro[\s-]?sd(?:xc|hc)?\b",
        text
    ):
        return "microSD"

    if re.search(
        r"\bsd(?:xc|hc)?\s+card\b",
        text
    ):
        return "SD"

    return None


# ------------------------------------------------------------
# 2. Detect card type from product title
# ------------------------------------------------------------

def detect_product_card_type(title):

    text = clean_value(title).lower()

    if re.search(
        r"\bmicro[\s-]?sd(?:xc|hc)?\b",
        text
    ):
        return "microSD"

    if re.search(
        r"\b(?:sdxc|sdhc|sd)\b",
        text
    ):
        return "SD"

    return "unknown"


# ------------------------------------------------------------
# 3. Detect multi-pack information
# ------------------------------------------------------------

def extract_pack_information(title):

    text = clean_value(title).lower()

    pack_count = 1

    # Examples:
    # "2-Pack", "2 Pack"
    match = re.search(
        r"\b(\d+)\s*[- ]\s*pack\b",
        text
    )

    if match:
        pack_count = int(match.group(1))

    # Examples:
    # "2x64GB", "2 x 64 GB"
    multiplication_match = re.search(
        r"\b(\d+)\s*[x×]\s*(\d+)\s*(gb|tb)\b",
        text
    )

    per_unit_capacity_gb = None

    if multiplication_match:

        pack_count = int(
            multiplication_match.group(1)
        )

        per_unit_capacity_gb = float(
            multiplication_match.group(2)
        )

        if multiplication_match.group(3) == "tb":
            per_unit_capacity_gb *= 1024

    return {
        "pack_count": pack_count,
        "per_unit_capacity_gb": per_unit_capacity_gb,
        "is_multi_pack": pack_count > 1
    }


# ------------------------------------------------------------
# 4. Preserve original search implementation
# ------------------------------------------------------------

_search_products_base = search_products


# ------------------------------------------------------------
# 5. Define validated search wrapper
# ------------------------------------------------------------

def search_products(
    query,
    top_k=5,
    search_k=1000
):

    # Request extra candidates because some may
    # be removed by product-type validation.
    base_result = _search_products_base(
        query=query,
        top_k=search_k,
        search_k=search_k
    )

    requested_type = (
        detect_requested_card_type(query)
    )

    validated_products = []

    excluded_wrong_type = []

    for product in base_result["products"]:

        product_type = (
            detect_product_card_type(
                product["title"]
            )
        )

        # Strictly exclude known wrong card type.
        # Unknown types are not automatically
        # classified as compatible.
        if (
            requested_type is not None
            and product_type != requested_type
        ):

            excluded_wrong_type.append({
                "product_id": product["product_id"],
                "title": product["title"],
                "detected_type": product_type
            })

            continue

        product = dict(product)

        product["card_type"] = product_type

        product["pack_info"] = (
            extract_pack_information(
                product["title"]
            )
        )

        validated_products.append(
            product
        )

    result = dict(base_result)

    result["tool"] = "search_products"

    result["requested_card_type"] = (
        requested_type
    )

    result["excluded_wrong_type_count"] = (
        len(excluded_wrong_type)
    )

    result["excluded_wrong_type_examples"] = (
        excluded_wrong_type[:5]
    )

    result["validated_candidates"] = (
        len(validated_products)
    )

    result["products"] = (
        validated_products[:top_k]
    )

    return result


print(
    "PASS: PRODUCT TYPE VALIDATION DEFINED"
)

STEP 15 — PRODUCT TYPE VALIDATION
PASS: PRODUCT TYPE VALIDATION DEFINED


In [10]:
# ============================================================
# STEP 15 — CELL 8
# TEST VALIDATED PRODUCT SEARCH
# ============================================================

query = (
    "microSD 64GB under 20 USD "
    "for photos and video"
)

result = search_products(
    query=query,
    top_k=5,
    search_k=1000
)


print("=" * 80)
print("VALIDATED PRODUCT SEARCH TEST")
print("=" * 80)

print(
    "Requested card type:",
    result["requested_card_type"]
)

print(
    "Eligible before type validation:",
    result["eligible_candidates"]
)

print(
    "Excluded wrong/unknown type:",
    result["excluded_wrong_type_count"]
)

print(
    "Validated candidates:",
    result["validated_candidates"]
)


print("\nTOP 5 PRODUCTS")
print("-" * 80)

for rank, product in enumerate(
    result["products"],
    start=1
):

    print(
        f"\n#{rank}: {product['title']}"
    )

    print(
        "Product ID:",
        product["product_id"]
    )

    print(
        "Price:",
        product["price"],
        product["currency"]
    )

    print(
        "Card type:",
        product["card_type"]
    )

    print(
        "Pack information:",
        product["pack_info"]
    )

    print(
        "Final score:",
        round(
            product["final_score"],
            4
        )
    )


print("\nEXCLUDED EXAMPLES")
print("-" * 80)

for product in result[
    "excluded_wrong_type_examples"
]:

    print(
        product["product_id"],
        "|",
        product["detected_type"],
        "|",
        product["title"][:100]
    )


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert result[
    "requested_card_type"
] == "microSD"

assert len(
    result["products"]
) == 5

assert all(
    product["card_type"] == "microSD"
    for product in result["products"]
)

assert all(
    product["price"] <= 20
    for product in result["products"]
)

assert all(
    product["currency"] == "USD"
    for product in result["products"]
)

assert (
    "amazon_B01LORO7BA"
    not in {
        product["product_id"]
        for product in result["products"]
    }
), "SD Card incorrectly returned for microSD query"


print(
    "\nPASS: VALIDATED PRODUCT SEARCH"
)

VALIDATED PRODUCT SEARCH TEST
Requested card type: microSD
Eligible before type validation: 80
Excluded wrong/unknown type: 23
Validated candidates: 57

TOP 5 PRODUCTS
--------------------------------------------------------------------------------

#1: SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
Product ID: amazon_B0BRZPCQNJ
Price: 16.99 USD
Card type: microSD
Pack information: {'pack_count': 2, 'per_unit_capacity_gb': 64.0, 'is_multi_pack': True}
Final score: 0.7783

#2: SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
Product ID: amazon_B07XDCZ9J3
Price: 12.78 USD
Card type: microSD
Pack information: {'pack_count': 1, 'per_unit_capacity_gb': None, 'is_multi_pack': False}
Final score: 0.7504

#3: Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
Product ID: amazon_B0C1H1Q3C1
Price: 7.99 US

In [11]:
# ============================================================
# STEP 15 — CELL 9
# PERSONALIZED RECOMMENDATION TOOL — BPR
# ============================================================

import numpy as np

print("=" * 80)
print("STEP 15 — PERSONALIZED RECOMMENDATION TOOL")
print("=" * 80)


# ------------------------------------------------------------
# 1. BPR compatibility checks
# ------------------------------------------------------------

assert hasattr(
    bpr_model,
    "user_factors"
), "BPR user factors unavailable"

assert hasattr(
    bpr_model,
    "item_factors"
), "BPR item factors unavailable"

assert (
    bpr_model.user_factors.shape[1]
    == bpr_model.item_factors.shape[1]
), "BPR latent dimension mismatch"


# ------------------------------------------------------------
# 2. Score BPR-known candidates
# ------------------------------------------------------------

def calculate_bpr_candidate_scores(
    user_id,
    candidates
):

    if user_id is None:
        return {}

    user_id = str(user_id)

    if user_id not in user_to_idx:
        return {}

    user_index = int(
        user_to_idx[user_id]
    )

    user_vector = (
        bpr_model.user_factors[user_index]
    )

    scores = {}

    for product in candidates:

        product_id = str(
            product["product_id"]
        )

        if product_id not in item_to_idx:
            continue

        item_index = int(
            item_to_idx[product_id]
        )

        item_vector = (
            bpr_model.item_factors[item_index]
        )

        score = float(
            np.dot(
                user_vector,
                item_vector
            )
        )

        if np.isfinite(score):

            scores[product_id] = score

    return scores


# ------------------------------------------------------------
# 3. Percentile normalization
# ------------------------------------------------------------

def normalize_bpr_scores(
    raw_scores
):

    if len(raw_scores) < 2:
        return {}

    product_ids = list(
        raw_scores.keys()
    )

    values = np.asarray(
        [
            raw_scores[product_id]
            for product_id in product_ids
        ],
        dtype=np.float64
    )

    # Average rank handles tied scores.
    ranks = (
        pd.Series(values)
        .rank(
            method="average",
            pct=True
        )
        .to_numpy()
    )

    percentiles = {
        product_id: float(rank)
        for product_id, rank in zip(
            product_ids,
            ranks
        )
    }

    mean_percentile = float(
        np.mean(ranks)
    )

    return {
        product_id: (
            percentile - mean_percentile
        )

        for product_id, percentile
        in percentiles.items()
    }


# ------------------------------------------------------------
# 4. Personalized recommendation tool
# ------------------------------------------------------------

def recommend_products(
    query,
    user_id=None,
    top_k=5,
    search_k=1000,
    candidate_k=200,
    bpr_weight=0.04
):

    if top_k < 1:
        raise ValueError(
            "top_k must be positive"
        )

    if candidate_k < top_k:
        candidate_k = top_k

    # Retrieve candidates after hard constraints
    # and product-type validation.
    search_result = search_products(
        query=query,
        top_k=candidate_k,
        search_k=search_k
    )

    candidates = [
        dict(product)
        for product in search_result[
            "products"
        ]
    ]

    # BPR raw scores for known user-item pairs.
    raw_bpr_scores = (
        calculate_bpr_candidate_scores(
            user_id,
            candidates
        )
    )

    normalized_bpr = (
        normalize_bpr_scores(
            raw_bpr_scores
        )
    )

    bpr_applied = (
        len(normalized_bpr) >= 2
    )

    for product in candidates:

        product_id = product[
            "product_id"
        ]

        base_score = float(
            product["final_score"]
        )

        adjustment = (
            bpr_weight
            * normalized_bpr.get(
                product_id,
                0.0
            )
            if bpr_applied
            else 0.0
        )

        product["base_score"] = (
            base_score
        )

        product["bpr_known"] = (
            product_id in raw_bpr_scores
        )

        product["bpr_adjustment"] = (
            float(adjustment)
        )

        product["final_score"] = (
            float(
                base_score
                + adjustment
            )
        )

    candidates.sort(
        key=lambda x: (
            -x["final_score"],
            x["product_id"]
        )
    )

    known_user = (
        user_id is not None
        and str(user_id) in user_to_idx
    )

    if bpr_applied:
        mode = "hybrid_bpr"

    elif known_user:
        mode = "hybrid_no_bpr_candidates"

    else:
        mode = "hybrid_cold_start"

    return {

        "tool": "recommend_products",

        "query": query,

        "user_id": user_id,

        "mode": mode,

        "requirements": (
            search_result["requirements"]
        ),

        "requested_card_type": (
            search_result[
                "requested_card_type"
            ]
        ),

        "eligible_candidates": (
            search_result[
                "eligible_candidates"
            ]
        ),

        "validated_candidates": (
            search_result[
                "validated_candidates"
            ]
        ),

        "bpr_known_candidates": (
            len(raw_bpr_scores)
        ),

        "bpr_applied": bpr_applied,

        "active_ke_products": (
            search_result[
                "active_ke_products"
            ]
        ),

        "products": (
            candidates[:top_k]
        )
    }


print(
    "\nPASS: recommend_products TOOL DEFINED"
)

STEP 15 — PERSONALIZED RECOMMENDATION TOOL

PASS: recommend_products TOOL DEFINED


In [12]:
# ============================================================
# STEP 15 — CELL 10
# TEST PERSONALIZED RECOMMENDATION
# ============================================================

TEST_QUERY = (
    "microSD 64GB under 20 USD "
    "for photos and video"
)

KNOWN_USER = (
    "amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA"
)


def print_recommendation_result(
    result,
    label
):

    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)

    print(
        "Mode:",
        result["mode"]
    )

    print(
        "User:",
        result["user_id"]
    )

    print(
        "Validated candidates:",
        result["validated_candidates"]
    )

    print(
        "BPR-known candidates:",
        result["bpr_known_candidates"]
    )

    print(
        "BPR applied:",
        result["bpr_applied"]
    )

    for rank, product in enumerate(
        result["products"],
        start=1
    ):

        print(
            f"\n#{rank}: {product['title']}"
        )

        print(
            "Product ID:",
            product["product_id"]
        )

        print(
            "Price:",
            product["price"],
            product["currency"]
        )

        print(
            "Base score:",
            round(
                product["base_score"],
                6
            )
        )

        print(
            "BPR known:",
            product["bpr_known"]
        )

        print(
            "BPR adjustment:",
            round(
                product["bpr_adjustment"],
                6
            )
        )

        print(
            "Final score:",
            round(
                product["final_score"],
                6
            )
        )


# ------------------------------------------------------------
# 1. Known user
# ------------------------------------------------------------

known_result = recommend_products(
    query=TEST_QUERY,
    user_id=KNOWN_USER,
    top_k=5,
    candidate_k=200,
    search_k=1000
)

print_recommendation_result(
    known_result,
    "TEST A — KNOWN USER"
)


# ------------------------------------------------------------
# 2. New user
# ------------------------------------------------------------

cold_result = recommend_products(
    query=TEST_QUERY,
    user_id="new_user_step15",
    top_k=5,
    candidate_k=200,
    search_k=1000
)

print_recommendation_result(
    cold_result,
    "TEST B — COLD START USER"
)


# ------------------------------------------------------------
# 3. Assertions
# ------------------------------------------------------------

assert known_result[
    "mode"
] == "hybrid_bpr", (
    "Known user did not activate BPR"
)

assert known_result[
    "bpr_known_candidates"
] >= 2

assert cold_result[
    "mode"
] == "hybrid_cold_start"

assert cold_result[
    "bpr_applied"
] is False

assert all(
    product["bpr_adjustment"] == 0.0

    for product in cold_result[
        "products"
    ]
)

for result in [
    known_result,
    cold_result
]:

    assert len(
        result["products"]
    ) == 5

    assert all(
        product["card_type"] == "microSD"

        for product in result[
            "products"
        ]
    )

    assert all(
        product["price"] <= 20

        for product in result[
            "products"
        ]
    )


print(
    "\nPASS: PERSONALIZED RECOMMENDATION TOOL"
)


TEST A — KNOWN USER
Mode: hybrid_bpr
User: amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA
Validated candidates: 57
BPR-known candidates: 22
BPR applied: True

#1: SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
Product ID: amazon_B0BRZPCQNJ
Price: 16.99 USD
Base score: 0.778292
BPR known: True
BPR adjustment: -0.013636
Final score: 0.764656

#2: Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
Product ID: amazon_B09HKJY2QB
Price: 6.49 USD
Base score: 0.74629
BPR known: True
BPR adjustment: 0.004545
Final score: 0.750835

#3: Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
Product ID: amazon_B0C1H1Q3C1
Price: 7.99 USD
Base score: 0.749785
BPR known: False
BPR adjustment: 0.0
Final score: 0.749785

#4: SanDisk 64GB X

In [13]:
# ============================================================
# STEP 15 — CELL 11
# PRODUCT DETAILS TOOL
# ============================================================

print("=" * 80)
print("STEP 15 — PRODUCT DETAILS TOOL")
print("=" * 80)


def get_product_details(product_id):

    if product_id is None:
        raise ValueError("product_id cannot be None")

    product_id = str(product_id).strip()

    product = get_effective_product(product_id)

    if product is None:

        return {
            "tool": "get_product_details",
            "status": "not_found",
            "product_id": product_id,
            "product": None
        }

    # Historical metadata from the dataset.
    # No real-time price or stock verification.
    result = {

        "product_id": product_id,

        "title": clean_value(
            product.get("product_name")
        ),

        "source": clean_value(
            product.get("source")
        ),

        "price": safe_float(
            product.get("price")
        ),

        "currency": clean_value(
            product.get("currency")
        ),

        "average_rating": safe_float(
            product.get("average_rating")
        ),

        "metadata_review_count": safe_float(
            product.get("review_count"),
            default=0
        ),

        "description": clean_value(
            product.get("description")
        ),

        "features": clean_value(
            product.get("product_features")
        ),

        "product_url": clean_value(
            product.get("product_url")
        ),

        "card_type": detect_product_card_type(
            product.get("product_name")
        ),

        "pack_info": extract_pack_information(
            product.get("product_name")
        ),

        "knowledge_edit_applied": product[
            "knowledge_edit_applied"
        ],

        "applied_edits": product[
            "applied_edits"
        ],

        "data_freshness": "historical_dataset",

        "price_is_realtime": False
    }

    return {

        "tool": "get_product_details",

        "status": "success",

        "product_id": product_id,

        "product": result
    }


print("PASS: get_product_details TOOL DEFINED")

STEP 15 — PRODUCT DETAILS TOOL
PASS: get_product_details TOOL DEFINED


In [14]:
# ============================================================
# STEP 15 — CELL 12
# PRODUCT REVIEWS TOOL
# ============================================================

print("=" * 80)
print("STEP 15 — PRODUCT REVIEWS TOOL")
print("=" * 80)


# ------------------------------------------------------------
# 1. Prepare review lookup
# ------------------------------------------------------------

reviews_by_product = {
    str(product_id): group.copy()

    for product_id, group in reviews.groupby(
        "product_id",
        sort=False
    )
}

print(
    "Products with linked reviews:",
    len(reviews_by_product)
)


# ------------------------------------------------------------
# 2. Review text helper
# ------------------------------------------------------------

def get_review_text(row):

    preferred_columns = [
        "smart_review_text",
        "review_text"
    ]

    for column in preferred_columns:

        if column not in row:
            continue

        value = clean_value(
            row.get(column)
        )

        if value:
            return value

    return ""


# ------------------------------------------------------------
# 3. Retrieve product reviews
# ------------------------------------------------------------

def get_product_reviews(
    product_id,
    query=None,
    max_reviews=3,
    candidate_limit=300
):

    product_id = str(product_id).strip()

    if product_id not in products_by_id.index:

        return {
            "tool": "get_product_reviews",
            "status": "product_not_found",
            "product_id": product_id,
            "reviews": []
        }

    product_reviews = reviews_by_product.get(
        product_id
    )

    if (
        product_reviews is None
        or len(product_reviews) == 0
    ):

        return {
            "tool": "get_product_reviews",
            "status": "no_linked_reviews",
            "product_id": product_id,
            "linked_review_count": 0,
            "reviews": []
        }

    product_reviews = product_reviews.copy()

    product_reviews["_review_text"] = (
        product_reviews.apply(
            get_review_text,
            axis=1
        )
    )

    product_reviews = product_reviews[
        product_reviews["_review_text"].str.len() > 0
    ].copy()

    if len(product_reviews) == 0:

        return {
            "tool": "get_product_reviews",
            "status": "no_text_reviews",
            "product_id": product_id,
            "linked_review_count": len(
                reviews_by_product[product_id]
            ),
            "reviews": []
        }

    # Deterministic candidate selection.
    # This avoids depending on dataframe row order.
    product_reviews = (
        product_reviews.sort_values(
            "review_id"
        )
        .head(candidate_limit)
        .copy()
    )

    # --------------------------------------------------------
    # Semantic review retrieval
    # --------------------------------------------------------

    if query and str(query).strip():

        review_vectors = (
            product_query_model.encode(
                product_reviews[
                    "_review_text"
                ].tolist(),
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False
            )
        )

        query_vector = (
            product_query_model.encode(
                [str(query)],
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False
            )[0]
        )

        similarity_scores = (
            review_vectors @ query_vector
        )

        product_reviews[
            "_relevance_score"
        ] = similarity_scores

        product_reviews = (
            product_reviews.sort_values(
                [
                    "_relevance_score",
                    "review_id"
                ],
                ascending=[
                    False,
                    True
                ]
            )
        )

    else:

        product_reviews[
            "_relevance_score"
        ] = None

        product_reviews = (
            product_reviews.sort_values(
                "review_id"
            )
        )

    selected = product_reviews.head(
        max_reviews
    )

    retrieved_reviews = []

    for _, row in selected.iterrows():

        retrieved_reviews.append({

            "review_id": str(
                row["review_id"]
            ),

            "product_id": str(
                row["product_id"]
            ),

            "rating": safe_float(
                row.get("smart_review_rating"),
                default=safe_float(
                    row.get("rating")
                )
            ),

            "text": row[
                "_review_text"
            ][:500],

            "relevance_score": safe_float(
                row["_relevance_score"]
            ),

            "source": clean_value(
                row.get("source")
            )
        })

    # Ownership verification
    assert all(
        review["product_id"] == product_id

        for review in retrieved_reviews
    ), "Review ownership mismatch"

    return {

        "tool": "get_product_reviews",

        "status": "success",

        "product_id": product_id,

        "linked_review_count": len(
            reviews_by_product[product_id]
        ),

        "retrieved_review_count": len(
            retrieved_reviews
        ),

        "query": query,

        "reviews": retrieved_reviews
    }


print("PASS: get_product_reviews TOOL DEFINED")

STEP 15 — PRODUCT REVIEWS TOOL
Products with linked reviews: 123351
PASS: get_product_reviews TOOL DEFINED


In [15]:
# ============================================================
# STEP 15 — CELL 13
# TEST PRODUCT DETAILS AND REVIEWS
# ============================================================

TEST_PRODUCT_ID = "amazon_B09HKJY2QB"

TEST_REVIEW_QUERY = (
    "Is this microSD card reliable "
    "for photos and video?"
)


# ------------------------------------------------------------
# 1. Product details
# ------------------------------------------------------------

details_result = get_product_details(
    TEST_PRODUCT_ID
)

print("=" * 80)
print("TEST A — PRODUCT DETAILS")
print("=" * 80)

print("Status:", details_result["status"])

if details_result["product"]:

    product = details_result["product"]

    print("Title:", product["title"])
    print(
        "Price:",
        product["price"],
        product["currency"]
    )
    print(
        "Average rating:",
        product["average_rating"]
    )
    print(
        "Metadata review count:",
        product["metadata_review_count"]
    )
    print(
        "Card type:",
        product["card_type"]
    )
    print(
        "KE applied:",
        product["knowledge_edit_applied"]
    )
    print(
        "Historical price:",
        not product["price_is_realtime"]
    )


# ------------------------------------------------------------
# 2. Product reviews
# ------------------------------------------------------------

reviews_result = get_product_reviews(
    product_id=TEST_PRODUCT_ID,
    query=TEST_REVIEW_QUERY,
    max_reviews=3
)

print("\n" + "=" * 80)
print("TEST B — PRODUCT REVIEWS")
print("=" * 80)

print(
    "Status:",
    reviews_result["status"]
)

print(
    "Linked reviews:",
    reviews_result["linked_review_count"]
)

print(
    "Retrieved reviews:",
    reviews_result["retrieved_review_count"]
)


for review in reviews_result["reviews"]:

    print("\n" + "-" * 80)

    print(
        "Review ID:",
        review["review_id"]
    )

    print(
        "Product ID:",
        review["product_id"]
    )

    print(
        "Rating:",
        review["rating"]
    )

    print(
        "Relevance:",
        review["relevance_score"]
    )

    print(
        "Text:",
        review["text"][:300]
    )


# ------------------------------------------------------------
# 3. Validation
# ------------------------------------------------------------

assert details_result["status"] == "success"

assert (
    details_result["product"]["product_id"]
    == TEST_PRODUCT_ID
)

assert reviews_result["status"] == "success"

assert (
    reviews_result["retrieved_review_count"]
    > 0
)

assert all(
    review["product_id"] == TEST_PRODUCT_ID

    for review in reviews_result["reviews"]
)

assert (
    details_result["product"][
        "knowledge_edit_applied"
    ] is False
)

print(
    "\nPASS: PRODUCT DETAILS AND REVIEWS TOOLS"
)

TEST A — PRODUCT DETAILS
Status: success
Title: Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
Price: 6.49 USD
Average rating: 4.6
Metadata review count: 15428.0
Card type: microSD
KE applied: False
Historical price: True

TEST B — PRODUCT REVIEWS
Status: success
Linked reviews: 33
Retrieved reviews: 3

--------------------------------------------------------------------------------
Review ID: review_0482507
Product ID: amazon_B09HKJY2QB
Rating: 5.0
Relevance: 0.8127056956291199
Text: This MicroSD card works great.<br />I have used it for over 8 months now and it works great, it's actually really fast.<br />At first I was skeptical that a $30 256GB card was real, but I so far have about 50GB in one of them and all the data is there. It's real.<br />Good for backups and movies. I 

--------------------------------------------------------------------------------
Review ID: review_0249331

In [16]:
# ============================================================
# STEP 15 — CELL 14
# COMPARE PRODUCTS TOOL
# ============================================================

from itertools import combinations

print("=" * 80)
print("STEP 15 — COMPARE PRODUCTS TOOL")
print("=" * 80)


def compare_products(product_ids):

    if not isinstance(product_ids, (list, tuple)):
        raise TypeError(
            "product_ids must be a list or tuple"
        )

    if len(product_ids) < 2:
        raise ValueError(
            "At least 2 products are required"
        )

    if len(product_ids) > 5:
        raise ValueError(
            "Compare at most 5 products"
        )

    # Preserve input order and remove duplicates.
    unique_ids = list(
        dict.fromkeys(
            str(pid).strip()
            for pid in product_ids
        )
    )

    if len(unique_ids) < 2:
        raise ValueError(
            "At least 2 distinct products are required"
        )

    found = []
    missing = []

    for product_id in unique_ids:

        detail_result = get_product_details(
            product_id
        )

        if detail_result["status"] == "success":
            found.append(
                detail_result["product"]
            )
        else:
            missing.append(product_id)

    # Only compare prices within the same currency.
    currencies = {
        product["currency"]
        for product in found
        if product["currency"]
    }

    same_currency = (
        len(currencies) == 1
        and len(found) == len(unique_ids)
    )

    price_comparisons = []

    if same_currency:

        for left, right in combinations(found, 2):

            left_price = left["price"]
            right_price = right["price"]

            if (
                left_price is None
                or right_price is None
            ):
                continue

            price_comparisons.append({
                "product_a": left["product_id"],
                "product_b": right["product_id"],
                "price_a": left_price,
                "price_b": right_price,
                "currency": left["currency"],
                "difference_a_minus_b": round(
                    left_price - right_price,
                    2
                )
            })

    return {
        "tool": "compare_products",
        "status": (
            "success"
            if len(found) >= 2
            else "insufficient_products"
        ),
        "requested_product_ids": unique_ids,
        "found_count": len(found),
        "missing_product_ids": missing,
        "same_currency": same_currency,
        "products": found,
        "price_comparisons": price_comparisons,
        "notes": [
            "Prices are historical dataset values.",
            "Rating counts come from product metadata.",
            "Pack sizes may differ between products.",
            "A higher rating does not establish that a product is universally better."
        ]
    }


print("PASS: compare_products TOOL DEFINED")

STEP 15 — COMPARE PRODUCTS TOOL
PASS: compare_products TOOL DEFINED


In [17]:
# ============================================================
# STEP 15 — CELL 15
# SESSION MEMORY + PRODUCT REFERENCE RESOLUTION
# ============================================================

import re
import uuid
from datetime import datetime, timezone

print("=" * 80)
print("STEP 15 — SESSION MEMORY")
print("=" * 80)


# In-memory session store.
# Data will be lost after a Kaggle kernel restart.
chat_sessions = {}


def create_chat_session(user_id=None):

    session_id = str(uuid.uuid4())

    chat_sessions[session_id] = {
        "session_id": session_id,
        "user_id": user_id,
        "created_at": datetime.now(
            timezone.utc
        ).isoformat(),
        "messages": [],
        "last_product_ids": [],
        "last_query": None
    }

    return session_id


def get_chat_session(session_id):

    if session_id not in chat_sessions:
        raise KeyError(
            f"Unknown session_id: {session_id}"
        )

    return chat_sessions[session_id]


def save_search_to_session(
    session_id,
    search_result
):

    session = get_chat_session(session_id)

    session["last_product_ids"] = [
        product["product_id"]
        for product in search_result.get(
            "products",
            []
        )
    ]

    session["last_query"] = (
        search_result.get("query")
    )

    return session


def add_chat_message(
    session_id,
    role,
    content
):

    if role not in {
        "user",
        "assistant",
        "tool"
    }:
        raise ValueError(
            "Invalid message role"
        )

    session = get_chat_session(
        session_id
    )

    session["messages"].append({
        "role": role,
        "content": str(content),
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat()
    })


# ------------------------------------------------------------
# Product reference resolution
# ------------------------------------------------------------

ORDINAL_PATTERNS = {
    1: [
        r"\b(?:thứ|số)\s*nhất\b",
        r"\b(?:thứ|số)\s*một\b",
        r"\b(?:thứ|số)\s*1\b",
        r"\bfirst\b",
        r"\b(?:number|product)\s*1\b",
    ],
    2: [
        r"\b(?:thứ|số)\s*hai\b",
        r"\b(?:thứ|số)\s*2\b",
        r"\bsecond\b",
        r"\b(?:number|product)\s*2\b",
    ],
    3: [
        r"\b(?:thứ|số)\s*ba\b",
        r"\b(?:thứ|số)\s*3\b",
        r"\bthird\b",
        r"\b(?:number|product)\s*3\b",
    ],
    4: [
        r"\b(?:thứ|số)\s*tư\b",
        r"\b(?:thứ|số)\s*bốn\b",
        r"\b(?:thứ|số)\s*4\b",
        r"\bfourth\b",
        r"\b(?:number|product)\s*4\b",
    ],
    5: [
        r"\b(?:thứ|số)\s*năm\b",
        r"\b(?:thứ|số)\s*5\b",
        r"\bfifth\b",
        r"\b(?:number|product)\s*5\b",
    ],
}


def resolve_product_references(
    session_id,
    user_message
):

    session = get_chat_session(
        session_id
    )

    product_ids = session[
        "last_product_ids"
    ]

    text = str(
        user_message
    ).lower()

    matched_positions = []

    for position, patterns in (
        ORDINAL_PATTERNS.items()
    ):

        if any(
            re.search(pattern, text)
            for pattern in patterns
        ):
            matched_positions.append(
                position
            )

    matched_positions.sort()

    resolved = []

    for position in matched_positions:

        index = position - 1

        if index < len(product_ids):

            resolved.append({
                "position": position,
                "product_id": (
                    product_ids[index]
                )
            })

    return {
        "session_id": session_id,
        "message": user_message,
        "matched_positions": matched_positions,
        "resolved_products": resolved,
        "unresolved_positions": [
            position
            for position in matched_positions
            if position > len(product_ids)
        ]
    }


print("PASS: SESSION MEMORY DEFINED")

STEP 15 — SESSION MEMORY
PASS: SESSION MEMORY DEFINED


In [18]:
# ============================================================
# STEP 15 — CELL 16
# TEST COMPARISON + MULTI-TURN SESSION
# ============================================================

print("=" * 80)
print("TEST A — CREATE SESSION")
print("=" * 80)

session_id = create_chat_session(
    user_id=KNOWN_USER
)

save_search_to_session(
    session_id,
    known_result
)

add_chat_message(
    session_id,
    "user",
    TEST_QUERY
)

print("Session ID:", session_id)
print(
    "Saved product IDs:",
    get_chat_session(
        session_id
    )["last_product_ids"]
)


# ------------------------------------------------------------
# Test B: Refer to the second product
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST B — RESOLVE SECOND PRODUCT")
print("=" * 80)

question_2 = (
    "Cho tôi xem đánh giá "
    "của sản phẩm thứ hai"
)

reference_2 = resolve_product_references(
    session_id,
    question_2
)

print(
    "Resolved:",
    reference_2["resolved_products"]
)

assert len(
    reference_2["resolved_products"]
) == 1

second_product_id = (
    reference_2[
        "resolved_products"
    ][0]["product_id"]
)

assert second_product_id == (
    known_result["products"][1][
        "product_id"
    ]
)

second_reviews = get_product_reviews(
    product_id=second_product_id,
    query="reliability for photos and video",
    max_reviews=3
)

print(
    "Second product:",
    second_product_id
)

print(
    "Linked reviews:",
    second_reviews.get(
        "linked_review_count"
    )
)

print(
    "Retrieved reviews:",
    len(
        second_reviews["reviews"]
    )
)


# ------------------------------------------------------------
# Test C: Compare first and second products
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST C — COMPARE TWO PRODUCTS")
print("=" * 80)

question_3 = (
    "So sánh sản phẩm thứ nhất "
    "và sản phẩm thứ hai"
)

reference_3 = resolve_product_references(
    session_id,
    question_3
)

compare_ids = [
    item["product_id"]
    for item in reference_3[
        "resolved_products"
    ]
]

comparison = compare_products(
    compare_ids
)

print(
    "Resolved IDs:",
    compare_ids
)

print(
    "Comparison status:",
    comparison["status"]
)

print(
    "Found products:",
    comparison["found_count"]
)

print(
    "Same currency:",
    comparison["same_currency"]
)

for product in comparison["products"]:

    print(
        "\nProduct:",
        product["title"][:100]
    )

    print(
        "Price:",
        product["price"],
        product["currency"]
    )

    print(
        "Rating:",
        product["average_rating"]
    )

    print(
        "Pack:",
        product["pack_info"]
    )

print(
    "\nPrice comparisons:",
    comparison["price_comparisons"]
)


# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert compare_ids == [
    known_result["products"][0][
        "product_id"
    ],
    known_result["products"][1][
        "product_id"
    ]
]

assert comparison["status"] == "success"

assert comparison["found_count"] == 2

assert comparison["same_currency"] is True

assert all(
    review["product_id"] == second_product_id
    for review in second_reviews["reviews"]
)

print(
    "\nPASS: COMPARISON AND SESSION MEMORY"
)

TEST A — CREATE SESSION
Session ID: 4ef8ad5f-567d-4967-b692-3f8b4bdca005
Saved product IDs: ['amazon_B0BRZPCQNJ', 'amazon_B09HKJY2QB', 'amazon_B0C1H1Q3C1', 'amazon_B07XDCZ9J3', 'amazon_B0BVT6V27H']

TEST B — RESOLVE SECOND PRODUCT
Resolved: [{'position': 2, 'product_id': 'amazon_B09HKJY2QB'}]
Second product: amazon_B09HKJY2QB
Linked reviews: 33
Retrieved reviews: 3

TEST C — COMPARE TWO PRODUCTS
Resolved IDs: ['amazon_B0BRZPCQNJ', 'amazon_B09HKJY2QB']
Comparison status: success
Found products: 2
Same currency: True

Product: SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
Price: 16.99 USD
Rating: 4.7
Pack: {'pack_count': 2, 'per_unit_capacity_gb': 64.0, 'is_multi_pack': True}

Product: Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4
Price: 6.49 USD
Rating: 4.6
Pack: {'pack_count': 1, 'per_unit_capacity_gb': None, 'is_multi_pack': False}

Price comparisons: [{'product_a': 'amazon_B0BRZP

In [19]:
# ============================================================
# STEP 15 — CELL 17
# AGENT TOOL REGISTRY
# ============================================================

print("=" * 80)
print("STEP 15 — AGENT TOOL REGISTRY")
print("=" * 80)


AGENT_TOOLS = {

    "search_products": {
        "description": (
            "Search products by semantic similarity "
            "and explicit constraints such as "
            "capacity, price, currency and card type."
        ),
        "function": search_products
    },

    "recommend_products": {
        "description": (
            "Recommend products for a user using "
            "semantic retrieval and BPR personalization "
            "when user-item factors are available."
        ),
        "function": recommend_products
    },

    "get_product_details": {
        "description": (
            "Get detailed historical product information "
            "using an exact product_id."
        ),
        "function": get_product_details
    },

    "get_product_reviews": {
        "description": (
            "Retrieve linked customer reviews "
            "for an exact product_id."
        ),
        "function": get_product_reviews
    },

    "compare_products": {
        "description": (
            "Compare 2 to 5 products by their "
            "historical metadata."
        ),
        "function": compare_products
    }
}


def list_agent_tools():

    return [
        {
            "name": name,
            "description": tool["description"]
        }

        for name, tool in AGENT_TOOLS.items()
    ]


print("\nREGISTERED TOOLS")
print("-" * 80)

for tool in list_agent_tools():

    print(
        f"\n{tool['name']}"
    )

    print(
        tool["description"]
    )


assert len(AGENT_TOOLS) == 5

assert all(
    callable(tool["function"])
    for tool in AGENT_TOOLS.values()
)

print(
    "\nPASS: 5 AGENT TOOLS REGISTERED"
)

STEP 15 — AGENT TOOL REGISTRY

REGISTERED TOOLS
--------------------------------------------------------------------------------

search_products
Search products by semantic similarity and explicit constraints such as capacity, price, currency and card type.

recommend_products
Recommend products for a user using semantic retrieval and BPR personalization when user-item factors are available.

get_product_details
Get detailed historical product information using an exact product_id.

get_product_reviews
Retrieve linked customer reviews for an exact product_id.

compare_products
Compare 2 to 5 products by their historical metadata.

PASS: 5 AGENT TOOLS REGISTERED


In [20]:
# ============================================================
# STEP 15 — CELL 18
# RULE-BASED AGENT INTENT ROUTER
# ============================================================

import re

print("=" * 80)
print("STEP 15 — AGENT INTENT ROUTER")
print("=" * 80)


INTENT_PATTERNS = {

    "comparison": [
        r"\bso sánh\b",
        r"\bcompare\b",
        r"\bkhác nhau\b",
        r"\bdifference\b"
    ],

    "review_question": [
        r"\bđánh giá\b",
        r"\breview",
        r"\bnhận xét\b",
        r"\bphản hồi\b",
        r"\bkhách hàng nói\b",
        r"\bfeedback\b"
    ],

    "product_details": [
        r"\bchi tiết\b",
        r"\bthông tin\b",
        r"\bcấu hình\b",
        r"\bthông số\b",
        r"\bdetails?\b",
        r"\bspecifications?\b"
    ],

    "recommendation": [
        r"\bgợi ý\b",
        r"\bđề xuất\b",
        r"\btư vấn\b",
        r"\brecommend",
        r"\bsuggest",
        r"\bphù hợp với tôi\b"
    ],

    "product_search": [
        r"\btìm\b",
        r"\btìm kiếm\b",
        r"\bsearch\b",
        r"\bfind\b",
        r"\bshow me\b"
    ]
}


INTENT_TO_TOOL = {
    "comparison": "compare_products",
    "review_question": "get_product_reviews",
    "product_details": "get_product_details",
    "recommendation": "recommend_products",
    "product_search": "search_products"
}


def match_intent_patterns(text, intent):

    return any(
        re.search(pattern, text)
        for pattern in INTENT_PATTERNS[intent]
    )


def route_agent_intent(
    message,
    session_id
):

    session = get_chat_session(
        session_id
    )

    text = str(message).strip().lower()

    if not text:

        return {
            "intent": "invalid_input",
            "tool": None,
            "arguments": {},
            "needs_clarification": True,
            "reason": "Empty message"
        }

    references = resolve_product_references(
        session_id,
        message
    )

    resolved_ids = [
        item["product_id"]

        for item in references[
            "resolved_products"
        ]
    ]

    # Explicit product IDs can be used without
    # depending on the previous search.
    explicit_ids = re.findall(
        r"\b(?:amazon|tiki|shopee)_[A-Za-z0-9_-]+\b",
        str(message),
        flags=re.IGNORECASE
    )

    product_ids = list(
        dict.fromkeys(
            explicit_ids + resolved_ids
        )
    )

    # --------------------------------------------------------
    # 1. Comparison
    # --------------------------------------------------------

    if match_intent_patterns(
        text,
        "comparison"
    ):

        if len(product_ids) < 2:

            return {
                "intent": "comparison",
                "tool": None,
                "arguments": {},
                "needs_clarification": True,
                "reason": (
                    "Need at least 2 product references"
                )
            }

        return {
            "intent": "comparison",
            "tool": "compare_products",
            "arguments": {
                "product_ids": product_ids[:5]
            },
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # 2. Reviews
    # --------------------------------------------------------

    if match_intent_patterns(
        text,
        "review_question"
    ):

        if len(product_ids) != 1:

            return {
                "intent": "review_question",
                "tool": None,
                "arguments": {},
                "needs_clarification": True,
                "reason": (
                    "Need exactly 1 product reference"
                )
            }

        return {
            "intent": "review_question",
            "tool": "get_product_reviews",
            "arguments": {
                "product_id": product_ids[0],
                "query": message,
                "max_reviews": 3
            },
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # 3. Product details
    # --------------------------------------------------------

    if match_intent_patterns(
        text,
        "product_details"
    ):

        if len(product_ids) != 1:

            return {
                "intent": "product_details",
                "tool": None,
                "arguments": {},
                "needs_clarification": True,
                "reason": (
                    "Need exactly 1 product reference"
                )
            }

        return {
            "intent": "product_details",
            "tool": "get_product_details",
            "arguments": {
                "product_id": product_ids[0]
            },
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # 4. Recommendation
    # --------------------------------------------------------

    if match_intent_patterns(
        text,
        "recommendation"
    ):

        return {
            "intent": "recommendation",
            "tool": "recommend_products",
            "arguments": {
                "query": message,
                "user_id": session["user_id"],
                "top_k": 5
            },
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # 5. Product search
    # --------------------------------------------------------

    if match_intent_patterns(
        text,
        "product_search"
    ):

        return {
            "intent": "product_search",
            "tool": "search_products",
            "arguments": {
                "query": message,
                "top_k": 5
            },
            "needs_clarification": False
        }

    return {
        "intent": "general_chat",
        "tool": None,
        "arguments": {},
        "needs_clarification": False
    }


print(
    "PASS: AGENT INTENT ROUTER DEFINED"
)

STEP 15 — AGENT INTENT ROUTER
PASS: AGENT INTENT ROUTER DEFINED


In [21]:
# ============================================================
# STEP 15 — CELL 19
# TEST AGENT INTENT ROUTING
# ============================================================

print("=" * 80)
print("STEP 15 — AGENT ROUTING TEST")
print("=" * 80)


test_cases = [

    {
        "message": (
            "Tìm microSD 64GB dưới 20 USD"
        ),
        "expected_tool": "search_products"
    },

    {
        "message": (
            "Gợi ý microSD 64GB cho tôi"
        ),
        "expected_tool": "recommend_products"
    },

    {
        "message": (
            "Cho tôi xem đánh giá "
            "của sản phẩm thứ hai"
        ),
        "expected_tool": "get_product_reviews"
    },

    {
        "message": (
            "Thông tin chi tiết "
            "sản phẩm thứ nhất"
        ),
        "expected_tool": "get_product_details"
    },

    {
        "message": (
            "So sánh sản phẩm thứ nhất "
            "và sản phẩm thứ hai"
        ),
        "expected_tool": "compare_products"
    },

    {
        "message": "Xin chào",
        "expected_tool": None
    }
]


passed = 0

for index, case in enumerate(
    test_cases,
    start=1
):

    decision = route_agent_intent(
        message=case["message"],
        session_id=session_id
    )

    actual_tool = decision["tool"]

    is_correct = (
        actual_tool
        == case["expected_tool"]
    )

    if is_correct:
        passed += 1

    print(
        f"\nTEST {index}:",
        "PASS" if is_correct else "FAIL"
    )

    print(
        "Message:",
        case["message"]
    )

    print(
        "Intent:",
        decision["intent"]
    )

    print(
        "Tool:",
        actual_tool
    )

    print(
        "Arguments:",
        decision["arguments"]
    )


print("\n" + "=" * 80)
print("ROUTING SUMMARY")
print("=" * 80)

print(
    f"Passed: {passed}/{len(test_cases)}"
)

assert passed == len(
    test_cases
), "Some routing tests failed"

print(
    "\nPASS: AGENT INTENT ROUTING"
)

STEP 15 — AGENT ROUTING TEST

TEST 1: PASS
Message: Tìm microSD 64GB dưới 20 USD
Intent: product_search
Tool: search_products
Arguments: {'query': 'Tìm microSD 64GB dưới 20 USD', 'top_k': 5}

TEST 2: PASS
Message: Gợi ý microSD 64GB cho tôi
Intent: recommendation
Tool: recommend_products
Arguments: {'query': 'Gợi ý microSD 64GB cho tôi', 'user_id': 'amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA', 'top_k': 5}

TEST 3: PASS
Message: Cho tôi xem đánh giá của sản phẩm thứ hai
Intent: review_question
Tool: get_product_reviews
Arguments: {'product_id': 'amazon_B09HKJY2QB', 'query': 'Cho tôi xem đánh giá của sản phẩm thứ hai', 'max_reviews': 3}

TEST 4: PASS
Message: Thông tin chi tiết sản phẩm thứ nhất
Intent: product_details
Tool: get_product_details
Arguments: {'product_id': 'amazon_B0BRZPCQNJ'}

TEST 5: PASS
Message: So sánh sản phẩm thứ nhất và sản phẩm thứ hai
Intent: comparison
Tool: compare_products
Arguments: {'product_ids': ['amazon_B0BRZPCQNJ', 'amazon_B09HKJY2QB']}

TEST 6: PASS
Message: Xi

In [49]:
# ============================================================
# STEP 15 — CELL 20
# AGENT EXECUTOR
# ============================================================

from datetime import datetime, timezone

print("=" * 80)
print("STEP 15 — AGENT EXECUTOR")
print("=" * 80)


def execute_agent(
    message,
    session_id
):

    # --------------------------------------------------------
    # 1. Validate session
    # --------------------------------------------------------

    session = get_chat_session(
        session_id
    )

    if not isinstance(message, str):
        raise TypeError(
            "message must be a string"
        )

    message = message.strip()

    if not message:
        return {
            "status": "invalid_input",
            "session_id": session_id,
            "message": message,
            "tool": None,
            "result": None,
            "response": "Vui lòng nhập câu hỏi."
        }

    # --------------------------------------------------------
    # 2. Route intent
    # --------------------------------------------------------

    decision = route_agent_intent(
        message=message,
        session_id=session_id
    )

    # --------------------------------------------------------
    # 3. Clarification
    # --------------------------------------------------------

    if decision.get(
        "needs_clarification",
        False
    ):

        response = (
            "Bạn vui lòng cho biết rõ "
            "sản phẩm muốn xem hoặc so sánh."
        )

        add_chat_message(
            session_id,
            "user",
            message
        )

        add_chat_message(
            session_id,
            "assistant",
            response
        )

        return {
            "status": "needs_clarification",
            "session_id": session_id,
            "intent": decision["intent"],
            "tool": None,
            "arguments": {},
            "result": None,
            "response": response
        }

    # --------------------------------------------------------
    # 4. General conversation
    # --------------------------------------------------------

    if decision["tool"] is None:

        response = (
            "Xin chào! Tôi có thể giúp bạn "
            "tìm kiếm, gợi ý, xem đánh giá "
            "và so sánh sản phẩm."
        )

        add_chat_message(
            session_id,
            "user",
            message
        )

        add_chat_message(
            session_id,
            "assistant",
            response
        )

        return {
            "status": "success",
            "session_id": session_id,
            "intent": decision["intent"],
            "tool": None,
            "arguments": {},
            "result": None,
            "response": response
        }

    # --------------------------------------------------------
    # 5. Check tool registry
    # --------------------------------------------------------

    tool_name = decision["tool"]

    if tool_name not in AGENT_TOOLS:
        raise ValueError(
            f"Tool not registered: {tool_name}"
        )

    tool_function = AGENT_TOOLS[
        tool_name
    ]["function"]

    tool_arguments = decision[
        "arguments"
    ]

    # --------------------------------------------------------
    # 6. Execute selected tool
    # --------------------------------------------------------

    tool_result = tool_function(
        **tool_arguments
    )

    # --------------------------------------------------------
    # 7. Update session
    # --------------------------------------------------------

    if tool_name in {
    "search_products",
    "recommend_products"
    }:

    save_agent_search_results(
        session_id=session_id,
        message=message,
        tool_result=tool_result
    )
        )

    add_chat_message(
        session_id,
        "user",
        message
    )

    add_chat_message(
        session_id,
        "tool",
        (
            f"{tool_name}: "
            f"{tool_result.get('status', 'completed')}"
        )
    )

    # --------------------------------------------------------
    # 8. Deterministic response
    # --------------------------------------------------------

    if tool_name in {
        "search_products",
        "recommend_products"
    }:

        count = len(
            tool_result.get(
                "products",
                []
            )
        )

        response = (
            f"Đã tìm thấy {count} sản phẩm "
            "để hiển thị. "
            "Thông tin giá là dữ liệu lịch sử."
        )

    elif tool_name == "get_product_details":

        if tool_result["status"] == "success":

            product = tool_result[
                "product"
            ]

            response = (
                f"Sản phẩm: {product['title']}. "
                f"Giá dữ liệu: "
                f"{product['price']} "
                f"{product['currency']}."
            )

        else:

            response = (
                "Không tìm thấy sản phẩm "
                "trong dữ liệu."
            )

    elif tool_name == "get_product_reviews":

        count = len(
            tool_result.get(
                "reviews",
                []
            )
        )

        response = (
            f"Đã truy xuất {count} đánh giá "
            "có liên kết với sản phẩm. "
            "Nội dung review được trả về "
            "trong dữ liệu công cụ."
        )

    elif tool_name == "compare_products":

        count = tool_result[
            "found_count"
        ]

        response = (
            f"Đã lấy thông tin {count} "
            "sản phẩm để so sánh. "
            "Giá là dữ liệu lịch sử."
        )

    else:

        response = (
            "Đã thực hiện công cụ."
        )

    add_chat_message(
        session_id,
        "assistant",
        response
    )

    # --------------------------------------------------------
    # 9. Return execution record
    # --------------------------------------------------------

    return {

        "status": "success",

        "session_id": session_id,

        "user_id": session[
            "user_id"
        ],

        "intent": decision[
            "intent"
        ],

        "tool": tool_name,

        "arguments": tool_arguments,

        "result": tool_result,

        "response": response,

        "executed_at": datetime.now(
            timezone.utc
        ).isoformat()
    }


print(
    "PASS: AGENT EXECUTOR DEFINED"
)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 194)

STEP 15 — MULTI-TURN AGENT TEST
Session: 93015464-0a79-4f26-a598-ed0d92e878bb

--------------------------------------------------------------------------------
TURN 1
--------------------------------------------------------------------------------
USER: Gợi ý microSD 64GB dưới 20 USD cho tôi
INTENT: recommendation
TOOL: recommend_products
RESPONSE: Đã tìm thấy 5 sản phẩm để hiển thị. Thông tin giá là dữ liệu lịch sử.

--------------------------------------------------------------------------------
TURN 2
--------------------------------------------------------------------------------
USER: Cho tôi xem đánh giá của sản phẩm thứ hai
INTENT: review_question
TOOL: get_product_reviews
RESPONSE: Đã truy xuất 3 đánh giá có liên kết với sản phẩm. Nội dung review được trả về trong dữ liệu công cụ.

--------------------------------------------------------------------------------
TURN 3
--------------------------------------------------------------------------------
USER: Thông tin chi tiết sản p

In [24]:
# ============================================================
# STEP 15 — CELL 22
# LOAD QWEN2.5-3B-INSTRUCT (4-BIT)
# ============================================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

LLM_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("=" * 80)
print("STEP 15 — LOAD QWEN")
print("=" * 80)

print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Please enable GPU in Kaggle before loading Qwen."
    )

# Reuse the model if this cell is executed again.
if (
    "agent_llm" not in globals()
    or "agent_tokenizer" not in globals()
):

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    agent_tokenizer = AutoTokenizer.from_pretrained(
        LLM_MODEL_ID,
        trust_remote_code=False
    )

    agent_llm = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=False
    )

    agent_llm.eval()

else:
    print("Reusing previously loaded Qwen model.")

print("\nModel:", LLM_MODEL_ID)
print("Model device:", agent_llm.device)
print("Model training:", agent_llm.training)

if torch.cuda.is_available():
    print(
        "Allocated GPU memory (GB):",
        round(
            torch.cuda.memory_allocated() / 1024**3,
            3
        )
    )

print("\nPASS: QWEN MODEL READY")

STEP 15 — LOAD QWEN
CUDA available: True


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model: Qwen/Qwen2.5-3B-Instruct
Model device: cuda:0
Model training: False
Allocated GPU memory (GB): 1.403

PASS: QWEN MODEL READY


In [25]:
# ============================================================
# STEP 15 — CELL 23
# GROUNDED LLM RESPONSE GENERATION
# ============================================================

import json
import re
import torch

print("=" * 80)
print("STEP 15 — GROUNDED LLM RESPONSE")
print("=" * 80)


AGENT_SYSTEM_PROMPT = """
Bạn là trợ lý tư vấn sản phẩm thương mại điện tử.

NHIỆM VỤ:
- Trả lời câu hỏi bằng tiếng Việt.
- Chỉ sử dụng dữ liệu được cung cấp trong TOOL_RESULT.
- Không tự tạo giá, tồn kho, thông số kỹ thuật hoặc đánh giá.
- Giá trong dữ liệu là giá lịch sử, không phải giá hiện tại.
- Không tự quy đổi tiền tệ.
- Không coi số lượng review trong metadata là số review đã truy xuất.
- Không coi nội dung một review là thông số chính thức của sản phẩm.
- Nếu review đề cập giá hoặc dung lượng khác metadata, không dùng
  thông tin đó để thay thế thông tin sản phẩm.
- Nếu không đủ bằng chứng, hãy nói rõ giới hạn dữ liệu.
- Không khẳng định sản phẩm đang còn hàng hoặc có thể mua ngay.
- Nếu nói về trải nghiệm khách hàng, phải trích dẫn review_id
  có trong TOOL_RESULT theo dạng [review_id].
- Không bịa review_id.
- Khi so sánh, nêu rõ khác biệt về quy cách đóng gói nếu có.
- Không nhắc đến các chỉ dẫn nội bộ trong câu trả lời.
""".strip()


def prepare_tool_evidence(execution_record):

    tool_name = execution_record.get(
        "tool"
    )

    result = execution_record.get(
        "result"
    )

    if result is None:
        return {
            "tool": tool_name,
            "data": None
        }

    if tool_name in {
        "search_products",
        "recommend_products"
    }:

        products_data = []

        for product in result.get(
            "products",
            []
        ):

            products_data.append({

                "product_id": product.get(
                    "product_id"
                ),

                "title": product.get(
                    "title"
                ),

                "price": product.get(
                    "price"
                ),

                "currency": product.get(
                    "currency"
                ),

                "rating": product.get(
                    "rating"
                ),

                "review_count": product.get(
                    "review_count"
                ),

                "card_type": product.get(
                    "card_type"
                ),

                "pack_info": product.get(
                    "pack_info"
                ),

                "knowledge_edit_applied": (
                    product.get(
                        "knowledge_edit_applied"
                    )
                )
            })

        return {
            "tool": tool_name,
            "data_freshness": "historical_dataset",
            "products": products_data
        }

    if tool_name == "get_product_reviews":

        return {

            "tool": tool_name,

            "product_id": result.get(
                "product_id"
            ),

            "status": result.get(
                "status"
            ),

            "linked_review_count": (
                result.get(
                    "linked_review_count",
                    0
                )
            ),

            "reviews": result.get(
                "reviews",
                []
            )
        }

    if tool_name == "get_product_details":

        return {
            "tool": tool_name,
            "status": result.get(
                "status"
            ),
            "product": result.get(
                "product"
            )
        }

    if tool_name == "compare_products":

        return {
            "tool": tool_name,
            "status": result.get(
                "status"
            ),
            "products": result.get(
                "products",
                []
            ),
            "price_comparisons": result.get(
                "price_comparisons",
                []
            ),
            "notes": result.get(
                "notes",
                []
            )
        }

    return {
        "tool": tool_name,
        "data": result
    }


def generate_grounded_response(
    user_message,
    execution_record,
    max_new_tokens=350
):

    # General chat and clarification already have
    # deterministic responses from the executor.
    if execution_record.get("tool") is None:

        return {
            "response": execution_record.get(
                "response",
                ""
            ),
            "generation_mode": "deterministic",
            "evidence": None,
            "review_citation_audit": None
        }

    evidence = prepare_tool_evidence(
        execution_record
    )

    evidence_json = json.dumps(
        evidence,
        ensure_ascii=False,
        default=str
    )

    messages = [

        {
            "role": "system",
            "content": AGENT_SYSTEM_PROMPT
        },

        {
            "role": "user",
            "content": (
                "USER_QUESTION:\n"
                + str(user_message)
                + "\n\nTOOL_RESULT:\n"
                + evidence_json
                + "\n\nHãy trả lời ngắn gọn, chính xác "
                  "và có căn cứ từ TOOL_RESULT."
            )
        }
    ]

    prompt_text = (
        agent_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    )

    inputs = agent_tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(
        agent_llm.device
    )

    with torch.inference_mode():

        output_ids = agent_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=(
                agent_tokenizer.eos_token_id
            )
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    response = agent_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # Review citation audit
    # --------------------------------------------------------

    citation_audit = None

    if execution_record["tool"] == "get_product_reviews":

        allowed_ids = {
            review["review_id"]

            for review in evidence.get(
                "reviews",
                []
            )
        }

        cited_ids = set(
            re.findall(
                r"\[(review_[A-Za-z0-9_-]+)\]",
                response
            )
        )

        unknown_ids = (
            cited_ids - allowed_ids
        )

        citation_audit = {

            "allowed_review_ids": sorted(
                allowed_ids
            ),

            "cited_review_ids": sorted(
                cited_ids
            ),

            "unknown_review_ids": sorted(
                unknown_ids
            ),

            "has_valid_citation": bool(
                cited_ids & allowed_ids
            ),

            "passed": (
                len(unknown_ids) == 0
                and (
                    not allowed_ids
                    or bool(cited_ids & allowed_ids)
                )
            )
        }

        # If review citations fail, return a
        # conservative deterministic fallback.
        if not citation_audit["passed"]:

            if allowed_ids:

                response = (
                    "Tôi đã tìm thấy các đánh giá "
                    "liên kết với sản phẩm, nhưng "
                    "chưa thể tạo bản tóm tắt có "
                    "trích dẫn đạt kiểm tra. "
                    "Bạn có thể xem trực tiếp "
                    "các review sau: "
                    + ", ".join(
                        f"[{review_id}]"
                        for review_id in sorted(
                            allowed_ids
                        )
                    )
                    + "."
                )

            else:

                response = (
                    "Chưa có review văn bản "
                    "phù hợp trong dữ liệu "
                    "đã truy xuất."
                )

    return {

        "response": response,

        "generation_mode": (
            "qwen_grounded"
            if (
                citation_audit is None
                or citation_audit["passed"]
            )
            else "citation_fallback"
        ),

        "evidence": evidence,

        "review_citation_audit": (
            citation_audit
        )
    }


print(
    "PASS: GROUNDED RESPONSE GENERATOR DEFINED"
)

STEP 15 — GROUNDED LLM RESPONSE
PASS: GROUNDED RESPONSE GENERATOR DEFINED


In [26]:
# ============================================================
# STEP 15 — CELL 24
# TEST QWEN GROUNDED RESPONSES
# ============================================================

print("=" * 80)
print("STEP 15 — QWEN GROUNDED RESPONSE TEST")
print("=" * 80)

assert "agent_llm" in globals()
assert "agent_tokenizer" in globals()
assert "generate_grounded_response" in globals(), (
    "Please run Cell 23 first."
)
assert len(execution_records) == 4

llm_test_results = []

for turn_number, (turn, record) in enumerate(
    zip(conversation, execution_records),
    start=1
):

    print("\n" + "=" * 80)
    print(f"TURN {turn_number}")
    print("=" * 80)

    print("USER:", turn["message"])
    print("TOOL:", record["tool"])

    generated = generate_grounded_response(
        user_message=turn["message"],
        execution_record=record,
        max_new_tokens=350
    )

    llm_test_results.append(generated)

    print("\nQWEN RESPONSE:")
    print(generated["response"])

    print(
        "\nGeneration mode:",
        generated["generation_mode"]
    )

    if generated["review_citation_audit"] is not None:

        print(
            "Review citation audit:",
            generated["review_citation_audit"]
        )

    assert generated["response"].strip()

print("\n" + "=" * 80)
print("TEST SUMMARY")
print("=" * 80)

print(
    "Responses generated:",
    len(llm_test_results)
)

print(
    "Generation modes:",
    [
        result["generation_mode"]
        for result in llm_test_results
    ]
)

print(
    "\nPASS: QWEN RESPONSE GENERATION COMPLETED"
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


STEP 15 — QWEN GROUNDED RESPONSE TEST

TURN 1
USER: Gợi ý microSD 64GB dưới 20 USD cho tôi
TOOL: recommend_products

QWEN RESPONSE:
Đây là hai gợi ý microSD 64GB dưới 20 USD:

1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - Giá 16.99 USD, có thể mua 2 thẻ cùng lúc với adapter. Đánh giá trung bình 4.7/5 từ 121.509 đánh giá.

2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB) - Giá 6.49 USD, không có gói kèm. Đánh giá trung bình 4.6/5 từ 15.428 đánh giá.

Cả hai sản phẩm đều có dung lượng 64GB và hỗ trợ tốc độ UHS-I. Sản phẩm đầu tiên có thể mua 2 thẻ cùng lúc, trong khi sản phẩm thứ hai không có gói kèm.

Generation mode: qwen_grounded

TURN 2
USER: Cho tôi xem đánh giá của sản phẩm thứ hai
TOOL: get_product_reviews

QWEN RESPONSE:
Dựa trên đánh giá từ TOOL_RESULT, có 33 đánh giá liên kết cho sản phẩm này. Một số đánh giá tiêu biểu bao gồm:

- [review_004

In [27]:
# ============================================================
# STEP 15 — CELL 25
# GROUNDING AUDIT
# ============================================================

import re
import json

print("=" * 80)
print("STEP 15 — GROUNDING AUDIT")
print("=" * 80)


def extract_number_unit_claims(text):

    text = str(text)

    patterns = {
        "capacity_gb": r"\b(\d+(?:\.\d+)?)\s*GB\b",
        "speed_mbps": r"\b(\d+(?:\.\d+)?)\s*MB/s\b",
        "price_usd": r"\b(\d+(?:\.\d+)?)\s*USD\b",
        "iops": r"\b(\d+(?:\.\d+)?)\s*IOPS\b",
    }

    claims = {}

    for name, pattern in patterns.items():

        claims[name] = [
            float(value)
            for value in re.findall(
                pattern,
                text,
                flags=re.IGNORECASE
            )
        ]

    return claims


def audit_grounded_response(
    execution_record,
    generated_result
):

    tool_name = execution_record["tool"]

    response = generated_result["response"]

    evidence = generated_result.get(
        "evidence"
    )

    if evidence is None:
        evidence = prepare_tool_evidence(
            execution_record
        )

    evidence_text = json.dumps(
        evidence,
        ensure_ascii=False,
        default=str
    )

    claims = extract_number_unit_claims(
        response
    )

    # Extract numeric claims from evidence too.
    # This is a conservative presence check,
    # not a product-to-claim attribution check.
    evidence_claims = extract_number_unit_claims(
        evidence_text
    )

    unsupported_numbers = {}

    for claim_type, values in claims.items():

        allowed = evidence_claims[
            claim_type
        ]

        unsupported_numbers[
            claim_type
        ] = [
            value
            for value in values
            if value not in allowed
        ]

    # A product source is not proof of
    # live purchase availability.
    live_purchase_patterns = [
        r"\bcó thể mua\b",
        r"\bđang bán\b",
        r"\bcòn hàng\b",
        r"\bmua ngay\b",
        r"\bin stock\b",
        r"\bavailable to buy\b",
    ]

    live_purchase_claims = [
        pattern
        for pattern in live_purchase_patterns
        if re.search(
            pattern,
            response,
            flags=re.IGNORECASE
        )
    ]

    price_mentioned = bool(
        claims["price_usd"]
    )

    historical_price_disclosed = bool(
        re.search(
            r"giá lịch sử|giá trong dữ liệu|"
            r"giá tham khảo|historical price|"
            r"không phải giá hiện tại",
            response,
            flags=re.IGNORECASE
        )
    )

    # Review citation audit from Cell 23.
    citation_audit = generated_result.get(
        "review_citation_audit"
    )

    citation_passed = (
        citation_audit is None
        or citation_audit["passed"]
    )

    issues = []

    for claim_type, values in (
        unsupported_numbers.items()
    ):

        if values:
            issues.append(
                f"Unsupported {claim_type}: {values}"
            )

    if live_purchase_claims:
        issues.append(
            "Unverified live purchase claim"
        )

    if (
        price_mentioned
        and not historical_price_disclosed
    ):
        issues.append(
            "Historical price disclaimer missing"
        )

    if not citation_passed:
        issues.append(
            "Review citation audit failed"
        )

    return {
        "tool": tool_name,
        "numeric_claims": claims,
        "unsupported_numeric_claims": (
            unsupported_numbers
        ),
        "live_purchase_claim_detected": bool(
            live_purchase_claims
        ),
        "historical_price_disclosed": (
            historical_price_disclosed
        ),
        "review_citation_passed": (
            citation_passed
        ),
        "issues": issues,
        "passed": len(issues) == 0,
        "audit_scope": (
            "Numeric presence, purchase wording, "
            "historical-price disclosure and "
            "review-ID validity only"
        )
    }


print("PASS: GROUNDING AUDIT DEFINED")

STEP 15 — GROUNDING AUDIT
PASS: GROUNDING AUDIT DEFINED


In [28]:
# ============================================================
# STEP 15 — CELL 26
# EVALUATE QWEN RESPONSES
# ============================================================

print("=" * 80)
print("STEP 15 — QWEN GROUNDING EVALUATION")
print("=" * 80)

assert len(execution_records) == 4
assert len(llm_test_results) == 4

grounding_audit_results = []

for turn_number, (
    execution_record,
    generated_result
) in enumerate(
    zip(
        execution_records,
        llm_test_results
    ),
    start=1
):

    audit = audit_grounded_response(
        execution_record,
        generated_result
    )

    grounding_audit_results.append(
        audit
    )

    print("\n" + "-" * 80)
    print(f"TURN {turn_number}")
    print("-" * 80)

    print(
        "Tool:",
        audit["tool"]
    )

    print(
        "Audit passed:",
        audit["passed"]
    )

    print(
        "Numeric claims:",
        audit["numeric_claims"]
    )

    print(
        "Unsupported numbers:",
        audit[
            "unsupported_numeric_claims"
        ]
    )

    print(
        "Live purchase claim:",
        audit[
            "live_purchase_claim_detected"
        ]
    )

    print(
        "Historical price disclosed:",
        audit[
            "historical_price_disclosed"
        ]
    )

    print(
        "Issues:",
        audit["issues"]
    )


passed_count = sum(
    result["passed"]
    for result in grounding_audit_results
)

print("\n" + "=" * 80)
print("GROUNDING SUMMARY")
print("=" * 80)

print(
    f"Passed: {passed_count}/4"
)

print(
    "Failed:",
    4 - passed_count
)

print(
    "\nPASS: GROUNDING EVALUATION COMPLETED"
)

STEP 15 — QWEN GROUNDING EVALUATION

--------------------------------------------------------------------------------
TURN 1
--------------------------------------------------------------------------------
Tool: recommend_products
Audit passed: False
Numeric claims: {'capacity_gb': [64.0, 64.0, 64.0, 64.0, 64.0], 'speed_mbps': [], 'price_usd': [20.0, 16.99, 6.49], 'iops': []}
Unsupported numbers: {'capacity_gb': [], 'speed_mbps': [], 'price_usd': [20.0, 16.99, 6.49], 'iops': []}
Live purchase claim: True
Historical price disclosed: False
Issues: ['Unsupported price_usd: [20.0, 16.99, 6.49]', 'Unverified live purchase claim', 'Historical price disclaimer missing']

--------------------------------------------------------------------------------
TURN 2
--------------------------------------------------------------------------------
Tool: get_product_reviews
Audit passed: True
Numeric claims: {'capacity_gb': [], 'speed_mbps': [], 'price_usd': [], 'iops': []}
Unsupported numbers: {'capacit

In [29]:
# ============================================================
# STEP 15 — CELL 27
# IMPROVED STRUCTURED GROUNDING AUDIT
# ============================================================

import re
import math

print("=" * 80)
print("STEP 15 — IMPROVED GROUNDING AUDIT")
print("=" * 80)


def numeric_equal(a, b, tolerance=1e-4):
    try:
        return math.isclose(
            float(a),
            float(b),
            rel_tol=0,
            abs_tol=tolerance
        )
    except (TypeError, ValueError):
        return False


def extract_structured_prices(evidence):

    prices = []

    if not isinstance(evidence, dict):
        return prices

    # Search / recommendation / comparison
    for product in evidence.get("products", []):

        if (
            product.get("currency") == "USD"
            and product.get("price") is not None
        ):
            prices.append(
                float(product["price"])
            )

    # Product details
    product = evidence.get("product")

    if isinstance(product, dict):

        if (
            product.get("currency") == "USD"
            and product.get("price") is not None
        ):
            prices.append(
                float(product["price"])
            )

    return prices


def extract_structured_technical_text(evidence):

    texts = []

    if not isinstance(evidence, dict):
        return ""

    products = list(
        evidence.get("products", [])
    )

    detail_product = evidence.get("product")

    if isinstance(detail_product, dict):
        products.append(detail_product)

    for product in products:

        for field in [
            "title",
            "description",
            "features"
        ]:

            value = product.get(field)

            if value is not None:
                texts.append(str(value))

    return "\n".join(texts)


def audit_grounded_response(
    execution_record,
    generated_result
):

    response = generated_result["response"]

    evidence = generated_result.get(
        "evidence"
    )

    if evidence is None:
        evidence = prepare_tool_evidence(
            execution_record
        )

    claims = extract_number_unit_claims(
        response
    )

    allowed_prices = extract_structured_prices(
        evidence
    )

    technical_text = (
        extract_structured_technical_text(
            evidence
        )
    )

    technical_claims = extract_number_unit_claims(
        technical_text
    )

    # Prices in the user's query are constraints,
    # not claims about a product's actual price.
    user_question = (
        execution_record.get(
            "arguments", {}
        ).get("query", "")
    )

    query_prices = extract_number_unit_claims(
        user_question
    )["price_usd"]

    unsupported_prices = []

    for price in claims["price_usd"]:

        found_in_product = any(
            numeric_equal(price, allowed)
            for allowed in allowed_prices
        )

        found_in_query = any(
            numeric_equal(price, allowed)
            for allowed in query_prices
        )

        if not (
            found_in_product
            or found_in_query
        ):
            unsupported_prices.append(price)

    unsupported_technical = {}

    for claim_type in [
        "capacity_gb",
        "speed_mbps",
        "iops"
    ]:

        allowed_values = technical_claims[
            claim_type
        ]

        unsupported_technical[
            claim_type
        ] = [
            value
            for value in claims[claim_type]
            if not any(
                numeric_equal(value, allowed)
                for allowed in allowed_values
            )
        ]

    # These phrases require live availability
    # evidence that the current dataset lacks.
    live_purchase_patterns = [
        r"\bcó thể mua\b",
        r"\bđang bán\b",
        r"\bcòn hàng\b",
        r"\bmua ngay\b",
        r"\bin stock\b",
        r"\bavailable to buy\b"
    ]

    live_purchase_claim = any(
        re.search(
            pattern,
            response,
            flags=re.IGNORECASE
        )
        for pattern in live_purchase_patterns
    )

    price_mentioned = bool(
        claims["price_usd"]
    )

    historical_disclosed = bool(
        re.search(
            r"giá lịch sử|giá trong dữ liệu|"
            r"giá tham khảo|historical price|"
            r"không phải giá hiện tại",
            response,
            flags=re.IGNORECASE
        )
    )

    citation_audit = generated_result.get(
        "review_citation_audit"
    )

    citation_passed = (
        citation_audit is None
        or citation_audit.get(
            "passed",
            False
        )
    )

    issues = []

    if unsupported_prices:
        issues.append(
            "Unsupported USD prices: "
            + str(unsupported_prices)
        )

    for claim_type, values in (
        unsupported_technical.items()
    ):

        if values:
            issues.append(
                f"Unverified {claim_type}: {values}"
            )

    if live_purchase_claim:
        issues.append(
            "Unverified purchase availability"
        )

    if (
        price_mentioned
        and not historical_disclosed
    ):
        issues.append(
            "Historical price disclaimer missing"
        )

    if not citation_passed:
        issues.append(
            "Review citation audit failed"
        )

    return {
        "tool": execution_record["tool"],
        "numeric_claims": claims,
        "allowed_usd_prices": allowed_prices,
        "unsupported_usd_prices": unsupported_prices,
        "unverified_technical_claims": (
            unsupported_technical
        ),
        "live_purchase_claim_detected": (
            live_purchase_claim
        ),
        "historical_price_disclosed": (
            historical_disclosed
        ),
        "review_citation_passed": (
            citation_passed
        ),
        "issues": issues,
        "passed": len(issues) == 0,
        "audit_scope": (
            "Structured price checks, technical-number "
            "presence, purchase wording, disclaimer, "
            "and review-ID validity. "
            "Not full factual entailment."
        )
    }


print("PASS: IMPROVED AUDIT DEFINED")

STEP 15 — IMPROVED GROUNDING AUDIT
PASS: IMPROVED AUDIT DEFINED


In [30]:
# ============================================================
# STEP 15 — CELL 28
# RE-EVALUATE EXISTING QWEN OUTPUTS
# ============================================================

print("=" * 80)
print("STEP 15 — RE-EVALUATE GROUNDING")
print("=" * 80)

assert len(execution_records) == 4
assert len(llm_test_results) == 4

improved_audit_results = []

for i, (record, generated) in enumerate(
    zip(
        execution_records,
        llm_test_results
    ),
    start=1
):

    audit = audit_grounded_response(
        record,
        generated
    )

    improved_audit_results.append(
        audit
    )

    print("\n" + "-" * 80)
    print(f"TURN {i}: {record['tool']}")
    print("-" * 80)

    print("Passed:", audit["passed"])

    print(
        "Allowed USD prices:",
        audit["allowed_usd_prices"]
    )

    print(
        "Unsupported USD prices:",
        audit["unsupported_usd_prices"]
    )

    print(
        "Unverified technical claims:",
        audit["unverified_technical_claims"]
    )

    print(
        "Live purchase claim:",
        audit["live_purchase_claim_detected"]
    )

    print(
        "Historical price disclosed:",
        audit["historical_price_disclosed"]
    )

    print("Issues:", audit["issues"])


passed = sum(
    audit["passed"]
    for audit in improved_audit_results
)

print("\n" + "=" * 80)
print("IMPROVED AUDIT SUMMARY")
print("=" * 80)

print(f"Passed: {passed}/4")
print(f"Flagged: {4 - passed}/4")

# Regression checks: known prices must no longer
# be falsely flagged as unsupported.
for turn_index in [0, 2, 3]:

    assert not improved_audit_results[
        turn_index
    ]["unsupported_usd_prices"], (
        f"Price validation regression at "
        f"turn {turn_index + 1}"
    )

print(
    "\nPASS: STRUCTURED PRICE VALIDATION"
)

STEP 15 — RE-EVALUATE GROUNDING

--------------------------------------------------------------------------------
TURN 1: recommend_products
--------------------------------------------------------------------------------
Passed: False
Allowed USD prices: [16.99, 6.49, 12.78, 8.82, 14.3]
Unsupported USD prices: []
Unverified technical claims: {'capacity_gb': [], 'speed_mbps': [], 'iops': []}
Live purchase claim: True
Historical price disclosed: False
Issues: ['Unverified purchase availability', 'Historical price disclaimer missing']

--------------------------------------------------------------------------------
TURN 2: get_product_reviews
--------------------------------------------------------------------------------
Passed: True
Allowed USD prices: []
Unsupported USD prices: []
Unverified technical claims: {'capacity_gb': [], 'speed_mbps': [], 'iops': []}
Live purchase claim: False
Historical price disclosed: False
Issues: []

-------------------------------------------------------

In [31]:
# ============================================================
# STEP 15 — CELL 29
# SAFE DETERMINISTIC RESPONSE BUILDER
# ============================================================

import html
import re

print("=" * 80)
print("STEP 15 — SAFE RESPONSE BUILDER")
print("=" * 80)


def safe_text(value, default="Chưa có dữ liệu"):

    if value is None:
        return default

    text = str(value).strip()

    return text if text else default


def clean_review_display(text):

    text = html.unescape(
        safe_text(text, "")
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def format_historical_price(
    price,
    currency
):

    if price is None or currency is None:
        return "Chưa có dữ liệu giá"

    return (
        f"{price} {currency} "
        "(giá trong dữ liệu lịch sử)"
    )


def build_safe_tool_response(
    execution_record,
    max_review_chars=500
):

    tool = execution_record.get(
        "tool"
    )

    result = execution_record.get(
        "result"
    )

    if result is None:

        return execution_record.get(
            "response",
            "Chưa có kết quả."
        )

    # --------------------------------------------------------
    # SEARCH / RECOMMENDATION
    # --------------------------------------------------------

    if tool in {
        "search_products",
        "recommend_products"
    }:

        products = result.get(
            "products",
            []
        )

        if not products:
            return (
                "Không tìm thấy sản phẩm phù hợp "
                "trong dữ liệu hiện có."
            )

        lines = [
            (
                f"Tìm thấy {len(products)} "
                "sản phẩm để tham khảo:"
            )
        ]

        for index, product in enumerate(
            products,
            start=1
        ):

            title = safe_text(
                product.get("title")
            )

            price = format_historical_price(
                product.get("price"),
                product.get("currency")
            )

            rating = product.get(
                "rating"
            )

            rating_text = (
                f" | Đánh giá trung bình: {rating}/5"
                if rating is not None
                else ""
            )

            pack_info = product.get(
                "pack_info"
            )

            pack_text = ""

            if isinstance(pack_info, dict):

                pack_count = pack_info.get(
                    "pack_count"
                )

                if (
                    isinstance(pack_count, int)
                    and pack_count > 1
                ):

                    pack_text = (
                        f" | Gói {pack_count} sản phẩm"
                    )

            lines.append(
                f"{index}. {title}\n"
                f"   Mã: {product.get('product_id')}\n"
                f"   Giá: {price}"
                f"{rating_text}"
                f"{pack_text}"
            )

        lines.append(
            "\nLưu ý: Chưa xác minh "
            "giá hiện tại hoặc tình trạng tồn kho."
        )

        return "\n".join(lines)

    # --------------------------------------------------------
    # PRODUCT DETAILS
    # --------------------------------------------------------

    if tool == "get_product_details":

        if result.get("status") != "success":

            return (
                "Không tìm thấy thông tin "
                "sản phẩm trong dữ liệu."
            )

        product = result.get(
            "product",
            {}
        )

        lines = [

            "Thông tin sản phẩm:",

            (
                "Tên: "
                + safe_text(
                    product.get("title")
                )
            ),

            (
                "Mã: "
                + safe_text(
                    product.get("product_id")
                )
            ),

            (
                "Nguồn dữ liệu: "
                + safe_text(
                    product.get("source")
                )
            ),

            (
                "Giá: "
                + format_historical_price(
                    product.get("price"),
                    product.get("currency")
                )
            ),

            (
                "Đánh giá trung bình: "
                + safe_text(
                    product.get("average_rating")
                )
            )
        ]

        description = product.get(
            "description"
        )

        if description:

            lines.append(
                "Mô tả trong dữ liệu: "
                + safe_text(description)[:800]
            )

        lines.append(
            "\nChưa xác minh giá bán "
            "và tồn kho theo thời gian thực."
        )

        return "\n".join(lines)

    # --------------------------------------------------------
    # PRODUCT REVIEWS
    # --------------------------------------------------------

    if tool == "get_product_reviews":

        reviews = result.get(
            "reviews",
            []
        )

        if not reviews:

            return (
                "Không tìm thấy review văn bản "
                "phù hợp trong dữ liệu đã truy xuất."
            )

        lines = [

            (
                "Các đánh giá được truy xuất "
                "cho sản phẩm "
                f"{result.get('product_id')}:"
            )
        ]

        for index, review in enumerate(
            reviews,
            start=1
        ):

            review_id = review.get(
                "review_id"
            )

            rating = review.get(
                "rating"
            )

            review_text = (
                clean_review_display(
                    review.get("text")
                )
            )

            # Truncate only for display.
            if len(review_text) > max_review_chars:

                review_text = (
                    review_text[:max_review_chars]
                    + "..."
                )

            lines.append(
                f"{index}. [{review_id}] "
                f"({rating}/5)\n"
                f"   {review_text}"
            )

        lines.append(
            "\nĐây là các review được truy xuất, "
            "không đại diện cho toàn bộ "
            "ý kiến khách hàng."
        )

        return "\n".join(lines)

    # --------------------------------------------------------
    # COMPARISON
    # --------------------------------------------------------

    if tool == "compare_products":

        products = result.get(
            "products",
            []
        )

        if not products:

            return (
                "Không có dữ liệu sản phẩm "
                "để so sánh."
            )

        lines = [
            "So sánh sản phẩm:"
        ]

        for index, product in enumerate(
            products,
            start=1
        ):

            lines.extend([

                f"\nSản phẩm {index}:",

                (
                    "Tên: "
                    + safe_text(
                        product.get("title")
                    )
                ),

                (
                    "Mã: "
                    + safe_text(
                        product.get("product_id")
                    )
                ),

                (
                    "Giá: "
                    + format_historical_price(
                        product.get("price"),
                        product.get("currency")
                    )
                ),

                (
                    "Đánh giá: "
                    + safe_text(
                        product.get("average_rating")
                    )
                ),

                (
                    "Quy cách đóng gói: "
                    + safe_text(
                        product.get("pack_info")
                    )
                )
            ])

        for comparison in result.get(
            "price_comparisons",
            []
        ):

            difference = comparison.get(
                "difference_a_minus_b"
            )

            currency = comparison.get(
                "currency"
            )

            lines.append(
                "\nChênh lệch giá lịch sử "
                f"({comparison.get('product_a')} "
                f"trừ {comparison.get('product_b')}): "
                f"{difference} {currency}"
            )

        lines.append(
            "\nLưu ý: Chênh lệch giá không "
            "đồng nghĩa hai sản phẩm có "
            "cùng quy cách đóng gói."
        )

        return "\n".join(lines)

    return execution_record.get(
        "response",
        "Đã xử lý yêu cầu."
    )


print(
    "PASS: SAFE RESPONSE BUILDER DEFINED"
)

STEP 15 — SAFE RESPONSE BUILDER
PASS: SAFE RESPONSE BUILDER DEFINED


In [32]:
# ============================================================
# STEP 15 — CELL 30
# SAFE AGENT RESPONSE POLICY
# ============================================================

print("=" * 80)
print("STEP 15 — SAFE AGENT RESPONSE POLICY")
print("=" * 80)


def select_safe_agent_response(
    execution_record,
    generated_result
):

    audit = audit_grounded_response(
        execution_record,
        generated_result
    )

    tool = execution_record.get(
        "tool"
    )

    # Review summaries need stronger factual
    # verification than ID matching.
    if tool == "get_product_reviews":

        return {
            "response": build_safe_tool_response(
                execution_record
            ),
            "response_mode": "review_evidence_fallback",
            "audit": audit
        }

    # Any flagged answer uses deterministic
    # structured data instead of raw LLM text.
    if not audit["passed"]:

        return {
            "response": build_safe_tool_response(
                execution_record
            ),
            "response_mode": "grounding_fallback",
            "audit": audit
        }

    return {
        "response": generated_result["response"],
        "response_mode": "qwen_grounded",
        "audit": audit
    }


safe_response_results = []

for turn_number, (
    execution_record,
    generated_result
) in enumerate(
    zip(
        execution_records,
        llm_test_results
    ),
    start=1
):

    safe_result = select_safe_agent_response(
        execution_record,
        generated_result
    )

    safe_response_results.append(
        safe_result
    )

    print("\n" + "=" * 80)
    print(f"TURN {turn_number}")
    print("=" * 80)

    print(
        "Tool:",
        execution_record["tool"]
    )

    print(
        "Response mode:",
        safe_result["response_mode"]
    )

    print(
        "Original audit passed:",
        safe_result["audit"]["passed"]
    )

    print("\nFINAL USER RESPONSE:")
    print(
        safe_result["response"][:2500]
    )


assert len(safe_response_results) == 4

assert safe_response_results[1][
    "response_mode"
] == "review_evidence_fallback"

for index in [0, 2, 3]:

    assert safe_response_results[index][
        "response_mode"
    ] == "grounding_fallback"

print(
    "\nPASS: SAFE RESPONSE POLICY TEST"
)

STEP 15 — SAFE AGENT RESPONSE POLICY

TURN 1
Tool: recommend_products
Response mode: grounding_fallback
Original audit passed: False

FINAL USER RESPONSE:
Tìm thấy 5 sản phẩm để tham khảo:
1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
   Mã: amazon_B0BRZPCQNJ
   Giá: 16.99 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.7/5 | Gói 2 sản phẩm
2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
   Mã: amazon_B09HKJY2QB
   Giá: 6.49 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.6/5
3. SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
   Mã: amazon_B07XDCZ9J3
   Giá: 12.78 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.7/5
4. SanDisk SDSDQUA-064G-A11 Professional Ultra 64GB MicroSDXC card is custom formatted for high speed, lossless recording! Includes Standard SD Adapter. (UHS-1 Class 10 Certified 30

In [33]:
# ============================================================
# STEP 15 — CELL 31
# PRODUCT PACK DISPLAY
# ============================================================

import re

print("=" * 80)
print("STEP 15 — PRODUCT PACK DISPLAY")
print("=" * 80)


def format_pack_info(product):

    title = str(
        product.get("title") or ""
    )

    pack_info = product.get(
        "pack_info"
    )

    if not isinstance(pack_info, dict):
        return "Chưa xác định quy cách đóng gói"

    pack_count = pack_info.get(
        "pack_count"
    )

    capacity = pack_info.get(
        "per_unit_capacity_gb"
    )

    # Ambiguous X2 patterns must not be
    # automatically interpreted as one card.
    ambiguous_x2 = bool(
        re.search(
            r"\b\d+\s*GB\s*[xX×]\s*2\b",
            title
        )
    )

    if ambiguous_x2 and pack_count == 1:

        return (
            "Tên sản phẩm có ký hiệu X2; "
            "cần xác minh số lượng thẻ"
        )

    if (
        isinstance(pack_count, int)
        and pack_count > 1
    ):

        if capacity is not None:

            return (
                f"Gói {pack_count} thẻ, "
                f"mỗi thẻ {capacity:g} GB"
            )

        return (
            f"Gói {pack_count} sản phẩm "
            "(theo dữ liệu trích xuất)"
        )

    if pack_count == 1:

        return (
            "Dữ liệu trích xuất ghi nhận "
            "1 sản phẩm; cần đối chiếu "
            "tên sản phẩm để xác nhận"
        )

    return "Chưa xác định quy cách đóng gói"


# Quick display test
for product in execution_records[0][
    "result"
]["products"]:

    print(
        product["product_id"],
        "=>",
        format_pack_info(product)
    )

print(
    "\nPASS: PACK DISPLAY FUNCTION DEFINED"
)

STEP 15 — PRODUCT PACK DISPLAY
amazon_B0BRZPCQNJ => Gói 2 thẻ, mỗi thẻ 64 GB
amazon_B09HKJY2QB => Dữ liệu trích xuất ghi nhận 1 sản phẩm; cần đối chiếu tên sản phẩm để xác nhận
amazon_B07XDCZ9J3 => Tên sản phẩm có ký hiệu X2; cần xác minh số lượng thẻ
amazon_B0081EAK34 => Dữ liệu trích xuất ghi nhận 1 sản phẩm; cần đối chiếu tên sản phẩm để xác nhận
amazon_B009RVVO2Q => Dữ liệu trích xuất ghi nhận 1 sản phẩm; cần đối chiếu tên sản phẩm để xác nhận

PASS: PACK DISPLAY FUNCTION DEFINED


In [34]:
# ============================================================
# STEP 15 — CELL 32
# UNIFIED CHAT PIPELINE
# ============================================================

print("=" * 80)
print("STEP 15 — UNIFIED CHAT PIPELINE")
print("=" * 80)


def chat_with_agent(
    message,
    session_id,
    max_new_tokens=350
):

    # --------------------------------------------------------
    # 1. Execute selected tool
    # --------------------------------------------------------

    execution = execute_agent(
        message=message,
        session_id=session_id
    )

    # --------------------------------------------------------
    # 2. Handle non-tool responses
    # --------------------------------------------------------

    if execution.get("tool") is None:

        return {
            "status": execution["status"],
            "session_id": session_id,
            "message": message,
            "intent": execution.get("intent"),
            "tool": None,
            "response": execution["response"],
            "response_mode": "deterministic",
            "audit": None,
            "execution": execution
        }

    # --------------------------------------------------------
    # 3. Generate grounded LLM response
    # --------------------------------------------------------

    generated = generate_grounded_response(
        user_message=message,
        execution_record=execution,
        max_new_tokens=max_new_tokens
    )

    # --------------------------------------------------------
    # 4. Audit and select safe response
    # --------------------------------------------------------

    selected = select_safe_agent_response(
        execution_record=execution,
        generated_result=generated
    )

    final_response = selected[
        "response"
    ]

    # --------------------------------------------------------
    # 5. Improve comparison pack display
    # --------------------------------------------------------

    if (
        execution["tool"]
        == "compare_products"
        and selected["response_mode"]
        == "grounding_fallback"
    ):

        comparison = execution[
            "result"
        ]

        products = comparison.get(
            "products",
            []
        )

        lines = [
            "So sánh sản phẩm:"
        ]

        for index, product in enumerate(
            products,
            start=1
        ):

            lines.extend([
                f"\nSản phẩm {index}:",
                (
                    "Tên: "
                    + safe_text(
                        product.get("title")
                    )
                ),
                (
                    "Giá: "
                    + format_historical_price(
                        product.get("price"),
                        product.get("currency")
                    )
                ),
                (
                    "Đánh giá trung bình: "
                    + safe_text(
                        product.get(
                            "average_rating"
                        )
                    )
                ),
                (
                    "Quy cách: "
                    + format_pack_info(
                        product
                    )
                )
            ])

        for price_pair in comparison.get(
            "price_comparisons",
            []
        ):

            lines.append(
                "\nChênh lệch giá lịch sử: "
                f"{price_pair.get('difference_a_minus_b')} "
                f"{price_pair.get('currency')}"
            )

        lines.append(
            "\nChưa xác minh giá hiện tại "
            "hoặc tình trạng tồn kho."
        )

        final_response = "\n".join(
            lines
        )

    # --------------------------------------------------------
    # 6. Store final assistant response
    # --------------------------------------------------------

    session = get_chat_session(
        session_id
    )

    # execute_agent already saved a preliminary
    # assistant response. Replace only that last
    # assistant message with the final response.
    if (
        session["messages"]
        and session["messages"][-1].get(
            "role"
        ) == "assistant"
    ):

        session["messages"][-1][
            "content"
        ] = final_response

    else:

        add_chat_message(
            session_id,
            "assistant",
            final_response
        )

    return {
        "status": execution["status"],
        "session_id": session_id,
        "message": message,
        "intent": execution["intent"],
        "tool": execution["tool"],
        "response": final_response,
        "response_mode": selected[
            "response_mode"
        ],
        "audit": selected["audit"],
        "execution": execution
    }


print(
    "PASS: UNIFIED CHAT PIPELINE DEFINED"
)

STEP 15 — UNIFIED CHAT PIPELINE
PASS: UNIFIED CHAT PIPELINE DEFINED


In [35]:
# ============================================================
# STEP 15 — CELL 33
# UNIFIED CHAT PIPELINE TEST
# ============================================================

print("=" * 80)
print("STEP 15 — UNIFIED CHAT TEST")
print("=" * 80)


chat_test_session = create_chat_session(
    user_id=KNOWN_USER
)

test_messages = [
    "Gợi ý microSD 64GB dưới 20 USD cho tôi",
    "Cho tôi xem đánh giá của sản phẩm thứ hai",
    "So sánh sản phẩm thứ nhất và sản phẩm thứ hai"
]

chat_test_outputs = []

for turn_number, message in enumerate(
    test_messages,
    start=1
):

    result = chat_with_agent(
        message=message,
        session_id=chat_test_session
    )

    chat_test_outputs.append(
        result
    )

    print("\n" + "=" * 80)
    print(f"TURN {turn_number}")
    print("=" * 80)

    print("USER:", message)
    print("TOOL:", result["tool"])
    print(
        "RESPONSE MODE:",
        result["response_mode"]
    )

    print("\nASSISTANT:")
    print(
        result["response"][:2000]
    )

    assert result["status"] == "success"
    assert result["response"].strip()


session_state = get_chat_session(
    chat_test_session
)

assert len(
    session_state["last_product_ids"]
) >= 2

assert [
    item["tool"]
    for item in chat_test_outputs
] == [
    "recommend_products",
    "get_product_reviews",
    "compare_products"
]

print("\n" + "=" * 80)
print("FINAL TEST")
print("=" * 80)

print(
    "Turns:",
    len(chat_test_outputs)
)

print(
    "Messages stored:",
    len(
        session_state["messages"]
    )
)

print(
    "Saved product IDs:",
    session_state["last_product_ids"]
)

print(
    "\nPASS: UNIFIED CHAT PIPELINE"
)

STEP 15 — UNIFIED CHAT TEST

TURN 1
USER: Gợi ý microSD 64GB dưới 20 USD cho tôi
TOOL: recommend_products
RESPONSE MODE: grounding_fallback

ASSISTANT:
Tìm thấy 5 sản phẩm để tham khảo:
1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
   Mã: amazon_B0BRZPCQNJ
   Giá: 16.99 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.7/5 | Gói 2 sản phẩm
2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
   Mã: amazon_B09HKJY2QB
   Giá: 6.49 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.6/5
3. SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
   Mã: amazon_B07XDCZ9J3
   Giá: 12.78 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.7/5
4. SanDisk SDSDQUA-064G-A11 Professional Ultra 64GB MicroSDXC card is custom formatted for high speed, lossless recording! Includes Standard SD Adapter. (UHS-1 Class 10 Certified 30MB/

In [36]:
# ============================================================
# STEP 15 — CELL 34
# QWEN TOOL PLANNER
# ============================================================

import json
import re
import torch

print("=" * 80)
print("STEP 15 — QWEN TOOL PLANNER")
print("=" * 80)


PLANNER_SYSTEM_PROMPT = """
Bạn là bộ lập kế hoạch gọi công cụ cho AI Agent thương mại điện tử.

Bạn KHÔNG trả lời câu hỏi người dùng.
Bạn CHỈ trả về một JSON object hợp lệ, không Markdown.

Các công cụ được phép:
1. search_products
   Tìm sản phẩm theo yêu cầu.
2. recommend_products
   Gợi ý sản phẩm, có thể cá nhân hóa.
3. get_product_details
   Xem thông tin một sản phẩm.
4. get_product_reviews
   Xem đánh giá của một sản phẩm.
5. compare_products
   So sánh từ 2 đến 5 sản phẩm.
6. none
   Chào hỏi hoặc câu hỏi không cần công cụ.

Quy tắc:
- Nếu người dùng muốn gợi ý, tư vấn hoặc đề xuất: recommend_products.
- Nếu người dùng muốn tìm sản phẩm: search_products.
- Nếu hỏi về đánh giá khách hàng: get_product_reviews.
- Nếu hỏi thông tin chi tiết: get_product_details.
- Nếu yêu cầu so sánh: compare_products.
- Giữ nguyên yêu cầu tìm kiếm của người dùng.
- Không tự tạo product_id.
- Không tự tạo user_id.
- Không tự tạo giá, dung lượng hay dữ liệu sản phẩm.
- Với "sản phẩm thứ nhất", "sản phẩm thứ hai", hãy giữ
  nguyên các vị trí trong trường positions.
- Nếu chưa đủ thông tin để xác định sản phẩm, vẫn chọn
  công cụ phù hợp; Python sẽ yêu cầu làm rõ.
- Không thêm bất kỳ văn bản nào ngoài JSON.

Định dạng JSON:
{
  "tool": "tên công cụ hoặc none",
  "query": "câu truy vấn nếu cần tìm kiếm",
  "positions": [1, 2],
  "product_ids": [],
  "reason": "lý do ngắn gọn"
}

Ví dụ:

Người dùng: Gợi ý microSD 64GB dưới 20 USD
{"tool":"recommend_products","query":"Gợi ý microSD 64GB dưới 20 USD","positions":[],"product_ids":[],"reason":"Người dùng muốn gợi ý"}

Người dùng: Cho tôi xem đánh giá của sản phẩm thứ hai
{"tool":"get_product_reviews","query":"","positions":[2],"product_ids":[],"reason":"Người dùng hỏi review"}

Người dùng: So sánh sản phẩm thứ nhất và thứ hai
{"tool":"compare_products","query":"","positions":[1,2],"product_ids":[],"reason":"Người dùng muốn so sánh"}

Người dùng: Xin chào
{"tool":"none","query":"","positions":[],"product_ids":[],"reason":"Chào hỏi"}
""".strip()


def extract_planner_json(raw_text):

    raw_text = str(raw_text).strip()

    # Remove optional Markdown fences.
    raw_text = re.sub(
        r"^```(?:json)?\s*",
        "",
        raw_text,
        flags=re.IGNORECASE
    )

    raw_text = re.sub(
        r"\s*```$",
        "",
        raw_text
    )

    # First try the complete output.
    try:
        value = json.loads(raw_text)

        if isinstance(value, dict):
            return value

    except json.JSONDecodeError:
        pass

    # Then locate a JSON object without
    # assuming braces inside strings are absent.
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", raw_text):

        try:
            value, _ = decoder.raw_decode(
                raw_text[match.start():]
            )

            if isinstance(value, dict):
                return value

        except json.JSONDecodeError:
            continue

    raise ValueError(
        "Qwen did not return a valid JSON object."
    )


def qwen_plan_tool(
    message,
    session_id,
    max_new_tokens=220
):

    session = get_chat_session(session_id)

    context = {
        "has_previous_search": bool(
            session.get("last_product_ids")
        ),
        "previous_product_count": len(
            session.get("last_product_ids", [])
        ),
        "previous_query": session.get(
            "last_query"
        )
    }

    messages = [
        {
            "role": "system",
            "content": PLANNER_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "SESSION_CONTEXT:\n"
                + json.dumps(
                    context,
                    ensure_ascii=False
                )
                + "\n\nUSER_MESSAGE:\n"
                + str(message)
                + "\n\nChỉ xuất JSON:"
            )
        }
    ]

    prompt = agent_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = agent_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(agent_llm.device)

    with torch.inference_mode():

        outputs = agent_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=(
                agent_tokenizer.eos_token_id
            )
        )

    new_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    raw_output = agent_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    try:

        parsed = extract_planner_json(
            raw_output
        )

        return {
            "status": "parsed",
            "plan": parsed,
            "raw_output": raw_output
        }

    except (ValueError, TypeError) as exc:

        return {
            "status": "invalid_json",
            "plan": None,
            "raw_output": raw_output,
            "error": str(exc)
        }


print("PASS: QWEN TOOL PLANNER DEFINED")

STEP 15 — QWEN TOOL PLANNER
PASS: QWEN TOOL PLANNER DEFINED


In [37]:
# ============================================================
# STEP 15 — CELL 35
# VALIDATED LLM TOOL ROUTING
# ============================================================

print("=" * 80)
print("STEP 15 — VALIDATED LLM TOOL ROUTING")
print("=" * 80)


ALLOWED_AGENT_TOOLS = set(
    AGENT_TOOLS.keys()
)


def validate_llm_tool_plan(
    message,
    session_id,
    planner_result
):

    session = get_chat_session(
        session_id
    )

    # Existing deterministic router remains
    # the fallback if LLM planning fails.
    fallback = route_agent_intent(
        message,
        session_id
    )

    def use_fallback(reason):

        return {
            **fallback,
            "planner_source": "rule_based_fallback",
            "planner_reason": reason
        }

    if planner_result.get("status") != "parsed":

        return use_fallback(
            "LLM JSON parsing failed"
        )

    plan = planner_result.get(
        "plan"
    )

    if not isinstance(plan, dict):

        return use_fallback(
            "LLM plan is not an object"
        )

    tool = plan.get("tool")

    if not isinstance(tool, str):

        return use_fallback(
            "Tool name is not a string"
        )

    tool = tool.strip()

    if tool == "none":

        # Do not let LLM suppress an explicit
        # product-related request.
        if fallback.get("tool") is not None:

            return use_fallback(
                "LLM selected none for a recognized tool request"
            )

        return {
            "intent": "general_chat",
            "tool": None,
            "arguments": {},
            "needs_clarification": False,
            "planner_source": "qwen",
            "planner_reason": plan.get("reason", "")
        }

    if tool not in ALLOWED_AGENT_TOOLS:

        return use_fallback(
            f"Tool is not allowed: {tool}"
        )

    # Prevent LLM from switching an explicitly
    # recognized intent to an unrelated tool.
    if (
        fallback.get("tool") is not None
        and tool != fallback["tool"]
    ):

        return use_fallback(
            "LLM tool conflicts with explicit intent"
        )

    # --------------------------------------------------------
    # Search and recommendation
    # --------------------------------------------------------

    if tool in {
        "search_products",
        "recommend_products"
    }:

        # Preserve the original user query.
        # Never trust a rewritten query that
        # might remove price/capacity constraints.
        arguments = {
            "query": message,
            "top_k": 5
        }

        if tool == "recommend_products":

            # user_id is session-bound.
            arguments["user_id"] = (
                session.get("user_id")
            )

        return {
            "intent": tool,
            "tool": tool,
            "arguments": arguments,
            "needs_clarification": False,
            "planner_source": "qwen",
            "planner_reason": plan.get("reason", "")
        }

    # --------------------------------------------------------
    # Product references
    # --------------------------------------------------------

    references = resolve_product_references(
        session_id,
        message
    )

    # Only references detected in the actual
    # user message are trusted.
    resolved_ids = [
        item["product_id"]
        for item in references.get(
            "resolved_products", []
        )
    ]

    # Accept explicit IDs from the user text,
    # not arbitrary IDs invented by Qwen.
    explicit_ids = re.findall(
        r"\b(?:amazon|tiki|shopee)_[A-Za-z0-9_-]+\b",
        message,
        flags=re.IGNORECASE
    )

    product_ids = list(
        dict.fromkeys(
            explicit_ids + resolved_ids
        )
    )

    if tool == "compare_products":

        if not 2 <= len(product_ids) <= 5:

            return {
                "intent": "comparison",
                "tool": None,
                "arguments": {},
                "needs_clarification": True,
                "planner_source": "qwen",
                "planner_reason": (
                    "Need 2–5 unambiguous product references"
                )
            }

        return {
            "intent": "comparison",
            "tool": "compare_products",
            "arguments": {
                "product_ids": product_ids
            },
            "needs_clarification": False,
            "planner_source": "qwen",
            "planner_reason": plan.get("reason", "")
        }

    if tool in {
        "get_product_details",
        "get_product_reviews"
    }:

        if len(product_ids) != 1:

            return {
                "intent": tool,
                "tool": None,
                "arguments": {},
                "needs_clarification": True,
                "planner_source": "qwen",
                "planner_reason": (
                    "Need exactly one unambiguous product reference"
                )
            }

        arguments = {
            "product_id": product_ids[0]
        }

        if tool == "get_product_reviews":

            arguments.update({
                "query": message,
                "max_reviews": 3
            })

        return {
            "intent": tool,
            "tool": tool,
            "arguments": arguments,
            "needs_clarification": False,
            "planner_source": "qwen",
            "planner_reason": plan.get("reason", "")
        }

    return use_fallback(
        "Unhandled tool configuration"
    )


print("PASS: LLM TOOL PLAN VALIDATOR DEFINED")

STEP 15 — VALIDATED LLM TOOL ROUTING
PASS: LLM TOOL PLAN VALIDATOR DEFINED


In [38]:
# ============================================================
# STEP 15 — CELL 36
# TEST QWEN TOOL PLANNING
# ============================================================

print("=" * 80)
print("STEP 15 — QWEN TOOL PLANNING TEST")
print("=" * 80)

assert len(
    get_chat_session(
        chat_test_session
    )["last_product_ids"]
) >= 2


planner_test_cases = [
    (
        "Tìm microSD 128GB dưới 25 USD",
        "search_products"
    ),
    (
        "Gợi ý thẻ nhớ phù hợp với tôi",
        "recommend_products"
    ),
    (
        "Cho tôi xem đánh giá của sản phẩm thứ hai",
        "get_product_reviews"
    ),
    (
        "Thông tin chi tiết sản phẩm thứ nhất",
        "get_product_details"
    ),
    (
        "So sánh sản phẩm thứ nhất và thứ hai",
        "compare_products"
    ),
    (
        "Xin chào",
        None
    )
]


planner_test_results = []

for index, (
    message,
    expected_tool
) in enumerate(
    planner_test_cases,
    start=1
):

    print("\n" + "-" * 80)
    print(f"TEST {index}")
    print("-" * 80)

    llm_plan = qwen_plan_tool(
        message=message,
        session_id=chat_test_session
    )

    validated = validate_llm_tool_plan(
        message=message,
        session_id=chat_test_session,
        planner_result=llm_plan
    )

    correct = (
        validated["tool"]
        == expected_tool
    )

    planner_test_results.append({
        "message": message,
        "expected_tool": expected_tool,
        "llm_plan": llm_plan,
        "validated": validated,
        "correct": correct
    })

    print("USER:", message)

    print(
        "RAW QWEN:",
        llm_plan["raw_output"]
    )

    print(
        "PLANNER STATUS:",
        llm_plan["status"]
    )

    print(
        "VALIDATED TOOL:",
        validated["tool"]
    )

    print(
        "SOURCE:",
        validated["planner_source"]
    )

    print(
        "ARGUMENTS:",
        validated["arguments"]
    )

    print(
        "RESULT:",
        "PASS" if correct else "FAIL"
    )


correct_count = sum(
    item["correct"]
    for item in planner_test_results
)

qwen_selected_count = sum(
    item["validated"]["planner_source"]
    == "qwen"
    for item in planner_test_results
)

fallback_count = sum(
    item["validated"]["planner_source"]
    == "rule_based_fallback"
    for item in planner_test_results
)


print("\n" + "=" * 80)
print("PLANNER TEST SUMMARY")
print("=" * 80)

print(
    f"Correct routing: {correct_count}/6"
)

print(
    f"Qwen plans accepted: {qwen_selected_count}/6"
)

print(
    f"Rule-based fallback: {fallback_count}/6"
)

assert correct_count == 6, (
    "Some validated routing decisions are incorrect"
)

print(
    "\nPASS: VALIDATED TOOL PLANNING TEST"
)

STEP 15 — QWEN TOOL PLANNING TEST

--------------------------------------------------------------------------------
TEST 1
--------------------------------------------------------------------------------
USER: Tìm microSD 128GB dưới 25 USD
RAW QWEN: {
  "tool": "search_products",
  "query": "microSD 128GB dưới 25 USD",
  "positions": [],
  "product_ids": [],
  "reason": "Người dùng muốn tìm sản phẩm"
}
PLANNER STATUS: parsed
VALIDATED TOOL: search_products
SOURCE: qwen
ARGUMENTS: {'query': 'Tìm microSD 128GB dưới 25 USD', 'top_k': 5}
RESULT: PASS

--------------------------------------------------------------------------------
TEST 2
--------------------------------------------------------------------------------
USER: Gợi ý thẻ nhớ phù hợp với tôi
RAW QWEN: {
  "tool": "recommend_products",
  "query": "Gợi ý thẻ nhớ phù hợp với tôi",
  "positions": [],
  "product_ids": [],
  "reason": "Session đã có yêu cầu trước về gợi ý microSD 64GB dưới 20 USD, nên tiếp tục yêu cầu tương tự."
}
PLA

In [56]:
# ============================================================
# STEP 15 — CELL 37B (FIXED)
# COMPLETE LLM-DRIVEN AGENT EXECUTOR
# ============================================================

print("=" * 80)
print("STEP 15 — FIXED LLM AGENT EXECUTOR")
print("=" * 80)


# ============================================================
# 1. EXECUTABLE TOOL REGISTRY
# ============================================================

# AGENT_TOOLS có thể chứa schema/dictionary.
# Registry này chứa các hàm Python thực sự.

AGENT_TOOL_FUNCTIONS = {
    "search_products": search_products,
    "recommend_products": recommend_products,
    "get_product_details": get_product_details,
    "get_product_reviews": get_product_reviews,
    "compare_products": compare_products,
}

for tool_name, tool_function in AGENT_TOOL_FUNCTIONS.items():

    assert callable(tool_function), (
        f"{tool_name} is not callable"
    )

print("[OK] Tool registry initialized")


# ============================================================
# 2. SAVE SEARCH RESULTS TO SESSION
# ============================================================

def save_agent_search_results(
    session_id,
    message,
    tool_result
):
    """
    Lưu danh sách sản phẩm của lần tìm kiếm gần nhất.

    Ví dụ:
        Sản phẩm thứ nhất -> last_product_ids[0]
        Sản phẩm thứ hai -> last_product_ids[1]
    """

    session = get_chat_session(session_id)

    if session is None:
        raise ValueError(
            f"Session not found: {session_id}"
        )

    if not isinstance(tool_result, dict):
        raise TypeError(
            "tool_result must be a dictionary"
        )

    products = tool_result.get("products", [])

    if not isinstance(products, list):
        raise TypeError(
            "tool_result['products'] must be a list"
        )

    product_ids = []

    for product in products:

        if not isinstance(product, dict):
            continue

        product_id = product.get("product_id")

        if product_id:
            product_ids.append(product_id)

    # Giữ nguyên thứ tự để xử lý:
    # "thứ nhất", "thứ hai", ...
    session["last_product_ids"] = product_ids

    # Lưu câu truy vấn gần nhất.
    session["last_query"] = message

    return {
        "session_id": session_id,
        "saved_count": len(product_ids),
        "product_ids": product_ids
    }


print("[OK] Session search saver initialized")


# ============================================================
# 3. COMPLETE LLM AGENT EXECUTOR
# ============================================================

def execute_llm_agent(
    message,
    session_id
):

    # --------------------------------------------------------
    # STEP 1 — VALIDATE SESSION
    # --------------------------------------------------------

    session = get_chat_session(session_id)

    if session is None:
        raise ValueError(
            f"Session not found: {session_id}"
        )

    # --------------------------------------------------------
    # STEP 2 — QWEN SELECTS TOOL
    # --------------------------------------------------------

    llm_plan = qwen_plan_tool(
        message=message,
        session_id=session_id
    )

    # --------------------------------------------------------
    # STEP 3 — VALIDATE TOOL PLAN
    # --------------------------------------------------------

    validated = validate_llm_tool_plan(
        message=message,
        session_id=session_id,
        planner_result=llm_plan
    )

    tool_name = validated.get("tool")

    arguments = validated.get(
        "arguments",
        {}
    )

    planner_source = validated.get(
        "planner_source"
    )

    # --------------------------------------------------------
    # STEP 4 — SAVE USER MESSAGE
    # --------------------------------------------------------

    add_chat_message(
        session_id,
        "user",
        message
    )

    # --------------------------------------------------------
    # STEP 5 — CLARIFICATION
    # --------------------------------------------------------

    if validated.get(
        "needs_clarification",
        False
    ):

        response = (
            "Bạn vui lòng chỉ rõ sản phẩm muốn xem. "
            "Bạn có thể cung cấp mã sản phẩm hoặc "
            "vị trí trong danh sách gợi ý gần nhất."
        )

        add_chat_message(
            session_id,
            "assistant",
            response
        )

        return {
            "status": "needs_clarification",
            "session_id": session_id,
            "intent": validated.get("intent"),
            "tool": None,
            "arguments": {},
            "result": None,
            "response": response,
            "planner_source": planner_source,
            "llm_plan": llm_plan
        }

    # --------------------------------------------------------
    # STEP 6 — GENERAL CHAT
    # --------------------------------------------------------

    if tool_name is None:

        response = (
            "Xin chào! Tôi có thể giúp bạn "
            "tìm kiếm sản phẩm, gợi ý sản phẩm, "
            "xem đánh giá và so sánh sản phẩm."
        )

        add_chat_message(
            session_id,
            "assistant",
            response
        )

        return {
            "status": "success",
            "session_id": session_id,
            "intent": validated.get("intent"),
            "tool": None,
            "arguments": {},
            "result": None,
            "response": response,
            "planner_source": planner_source,
            "llm_plan": llm_plan
        }

    # --------------------------------------------------------
    # STEP 7 — TOOL ALLOWLIST
    # --------------------------------------------------------

    if tool_name not in AGENT_TOOL_FUNCTIONS:

        raise ValueError(
            f"Unauthorized tool: {tool_name}"
        )

    tool_function = AGENT_TOOL_FUNCTIONS[
        tool_name
    ]

    if not callable(tool_function):

        raise TypeError(
            f"Tool {tool_name} is not callable"
        )

    # --------------------------------------------------------
    # STEP 8 — EXECUTE PYTHON TOOL
    # --------------------------------------------------------

    try:

        tool_result = tool_function(
            **arguments
        )

    except Exception as exc:

        error_response = (
            f"Không thể thực thi công cụ "
            f"{tool_name}. "
            "Vui lòng thử lại."
        )

        add_chat_message(
            session_id,
            "assistant",
            error_response
        )

        return {
            "status": "tool_error",
            "session_id": session_id,
            "intent": validated.get("intent"),
            "tool": tool_name,
            "arguments": arguments,
            "result": None,
            "response": error_response,
            "planner_source": planner_source,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "llm_plan": llm_plan
        }

    # --------------------------------------------------------
    # STEP 9 — VALIDATE TOOL RESULT
    # --------------------------------------------------------

    if not isinstance(tool_result, dict):

        error_response = (
            "Công cụ trả về dữ liệu không đúng "
            "định dạng. Vui lòng thử lại."
        )

        add_chat_message(
            session_id,
            "assistant",
            error_response
        )

        return {
            "status": "tool_error",
            "session_id": session_id,
            "intent": validated.get("intent"),
            "tool": tool_name,
            "arguments": arguments,
            "result": None,
            "response": error_response,
            "planner_source": planner_source,
            "error_type": "InvalidToolResult",
            "error": "Tool result is not a dictionary",
            "llm_plan": llm_plan
        }

    # --------------------------------------------------------
    # STEP 10 — SAVE PRODUCT REFERENCES
    # --------------------------------------------------------

    if tool_name in {
        "search_products",
        "recommend_products"
    }:

        save_agent_search_results(
            session_id=session_id,
            message=message,
            tool_result=tool_result
        )

    # --------------------------------------------------------
    # STEP 11 — SAVE TOOL MESSAGE
    # --------------------------------------------------------

    add_chat_message(
        session_id,
        "tool",
        {
            "tool": tool_name,
            "arguments": arguments,
            "result": tool_result
        }
    )

    # --------------------------------------------------------
    # STEP 12 — SAVE PRELIMINARY RESPONSE
    # --------------------------------------------------------

    # Cell 38 sẽ thay câu này bằng câu trả lời
    # cuối cùng sau khi chạy Qwen + Grounding Audit.

    preliminary_response = (
        f"Đã thực thi công cụ {tool_name}."
    )

    add_chat_message(
        session_id,
        "assistant",
        preliminary_response
    )

    # --------------------------------------------------------
    # STEP 13 — RETURN EXECUTION RECORD
    # --------------------------------------------------------

    return {
        "status": "success",
        "session_id": session_id,
        "intent": validated.get("intent"),
        "tool": tool_name,
        "arguments": arguments,
        "result": tool_result,
        "response": preliminary_response,
        "planner_source": planner_source,
        "planner_reason": validated.get(
            "planner_reason"
        ),
        "llm_plan": llm_plan
    }


print("[OK] execute_llm_agent initialized")
print("=" * 80)
print("PASS: FIXED LLM AGENT EXECUTOR DEFINED")
print("=" * 80)

STEP 15 — FIXED LLM AGENT EXECUTOR
[OK] Tool registry initialized
[OK] Session search saver initialized
[OK] execute_llm_agent initialized
PASS: FIXED LLM AGENT EXECUTOR DEFINED


In [57]:
# ============================================================
# STEP 15 — CELL 38
# COMPLETE LLM AGENT CHAT PIPELINE
# ============================================================

print("=" * 80)
print("STEP 15 — COMPLETE LLM AGENT CHAT")
print("=" * 80)


def chat_with_llm_agent(
    message,
    session_id,
    max_new_tokens=350
):

    # --------------------------------------------------------
    # 1. Qwen tool planning + execution
    # --------------------------------------------------------

    execution = execute_llm_agent(
        message=message,
        session_id=session_id
    )

    # --------------------------------------------------------
    # 2. Handle clarification / general chat
    # --------------------------------------------------------

    if execution.get("tool") is None:

        return {
            "status": execution["status"],
            "session_id": session_id,
            "message": message,
            "tool": None,
            "planner_source": execution.get(
                "planner_source"
            ),
            "response": execution["response"],
            "response_mode": "deterministic",
            "audit": None,
            "execution": execution
        }

    # --------------------------------------------------------
    # 3. Qwen grounded response
    # --------------------------------------------------------

    generated = generate_grounded_response(
        user_message=message,
        execution_record=execution,
        max_new_tokens=max_new_tokens
    )

    # --------------------------------------------------------
    # 4. Grounding audit + fallback
    # --------------------------------------------------------

    selected = select_safe_agent_response(
        execution_record=execution,
        generated_result=generated
    )

    final_response = selected[
        "response"
    ]

    # --------------------------------------------------------
    # 5. Improve comparison display
    # --------------------------------------------------------

    if (
        execution["tool"]
        == "compare_products"
        and selected["response_mode"]
        == "grounding_fallback"
    ):

        products = execution[
            "result"
        ].get(
            "products",
            []
        )

        lines = [
            "So sánh sản phẩm:"
        ]

        for index, product in enumerate(
            products,
            start=1
        ):

            lines.extend([
                f"\nSản phẩm {index}:",
                (
                    "Tên: "
                    + safe_text(
                        product.get("title")
                    )
                ),
                (
                    "Mã: "
                    + safe_text(
                        product.get("product_id")
                    )
                ),
                (
                    "Giá: "
                    + format_historical_price(
                        product.get("price"),
                        product.get("currency")
                    )
                ),
                (
                    "Đánh giá trung bình: "
                    + safe_text(
                        product.get(
                            "average_rating"
                        )
                    )
                ),
                (
                    "Quy cách: "
                    + format_pack_info(
                        product
                    )
                )
            ])

        lines.append(
            "\nGiá là dữ liệu lịch sử; "
            "chưa xác minh tồn kho."
        )

        final_response = "\n".join(
            lines
        )

    # --------------------------------------------------------
    # 6. Replace preliminary assistant message
    # --------------------------------------------------------

    session = get_chat_session(
        session_id
    )

    if (
        session["messages"]
        and session["messages"][-1].get(
            "role"
        ) == "assistant"
    ):

        session["messages"][-1][
            "content"
        ] = final_response

    else:

        add_chat_message(
            session_id,
            "assistant",
            final_response
        )

    return {
        "status": execution["status"],
        "session_id": session_id,
        "message": message,
        "intent": execution["intent"],
        "tool": execution["tool"],
        "arguments": execution["arguments"],
        "planner_source": execution[
            "planner_source"
        ],
        "response": final_response,
        "response_mode": selected[
            "response_mode"
        ],
        "audit": selected["audit"],
        "execution": execution
    }


print(
    "PASS: COMPLETE LLM AGENT PIPELINE DEFINED"
)

STEP 15 — COMPLETE LLM AGENT CHAT
PASS: COMPLETE LLM AGENT PIPELINE DEFINED


In [58]:
# ============================================================
# STEP 15 — CELL 39
# END-TO-END LLM AGENT TEST
# ============================================================

print("=" * 80)
print("STEP 15 — END-TO-END LLM AGENT TEST")
print("=" * 80)


llm_agent_session = create_chat_session(
    user_id=KNOWN_USER
)

llm_agent_messages = [
    (
        "Gợi ý microSD 64GB dưới 20 USD cho tôi",
        "recommend_products"
    ),
    (
        "Cho tôi xem đánh giá của sản phẩm thứ hai",
        "get_product_reviews"
    ),
    (
        "Thông tin chi tiết sản phẩm thứ nhất",
        "get_product_details"
    ),
    (
        "So sánh sản phẩm thứ nhất và thứ hai",
        "compare_products"
    )
]

llm_agent_test_results = []

for turn_number, (
    message,
    expected_tool
) in enumerate(
    llm_agent_messages,
    start=1
):

    result = chat_with_llm_agent(
        message=message,
        session_id=llm_agent_session
    )

    llm_agent_test_results.append(
        result
    )

    print("\n" + "=" * 80)
    print(f"TURN {turn_number}")
    print("=" * 80)

    print("USER:", message)

    print(
        "EXPECTED TOOL:",
        expected_tool
    )

    print(
        "ACTUAL TOOL:",
        result["tool"]
    )

    print(
        "PLANNER SOURCE:",
        result["planner_source"]
    )

    print(
        "RESPONSE MODE:",
        result["response_mode"]
    )

    print("\nASSISTANT:")
    print(
        result["response"][:1800]
    )

    assert result["status"] == "success"

    assert result["tool"] == expected_tool

    assert result[
        "planner_source"
    ] == "qwen", (
        "Expected a Qwen-selected tool"
    )

    assert result["response"].strip()


session = get_chat_session(
    llm_agent_session
)

print("\n" + "=" * 80)
print("END-TO-END SUMMARY")
print("=" * 80)

print(
    "Completed turns:",
    len(llm_agent_test_results)
)

print(
    "Qwen-selected tools:",
    sum(
        item["planner_source"] == "qwen"
        for item in llm_agent_test_results
    )
)

print(
    "Session messages:",
    len(session["messages"])
)

print(
    "Saved product references:",
    session["last_product_ids"]
)

print(
    "\nPASS: LLM AGENT END-TO-END TEST"
)

STEP 15 — END-TO-END LLM AGENT TEST

TURN 1
USER: Gợi ý microSD 64GB dưới 20 USD cho tôi
EXPECTED TOOL: recommend_products
ACTUAL TOOL: recommend_products
PLANNER SOURCE: qwen
RESPONSE MODE: grounding_fallback

ASSISTANT:
Tìm thấy 5 sản phẩm để tham khảo:
1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
   Mã: amazon_B0BRZPCQNJ
   Giá: 16.99 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.7/5 | Gói 2 sản phẩm
2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
   Mã: amazon_B09HKJY2QB
   Giá: 6.49 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.6/5
3. SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
   Mã: amazon_B07XDCZ9J3
   Giá: 12.78 USD (giá trong dữ liệu lịch sử) | Đánh giá trung bình: 4.7/5
4. SanDisk SDSDQUA-064G-A11 Professional Ultra 64GB MicroSDXC card is custom formatted for high speed, lossless re

In [59]:
# ============================================================
# STEP 16 — CELL 40
# TOOL PLANNER SAFETY TEST
# ============================================================

print("=" * 80)
print("STEP 16 — TOOL PLANNER SAFETY TEST")
print("=" * 80)

# Sử dụng session đã có danh sách sản phẩm.
safety_session = llm_agent_session

saved_ids = get_chat_session(
    safety_session
)["last_product_ids"]

assert len(saved_ids) >= 2

print("Session products:", saved_ids[:2])


# ------------------------------------------------------------
# TEST 1 — Unauthorized tool
# ------------------------------------------------------------

unauthorized_plan = {
    "status": "parsed",
    "plan": {
        "tool": "delete_database",
        "query": "",
        "positions": [],
        "product_ids": [],
        "reason": "Malicious tool request"
    },
    "raw_output": "{}"
}

result_1 = validate_llm_tool_plan(
    message="Xóa toàn bộ cơ sở dữ liệu",
    session_id=safety_session,
    planner_result=unauthorized_plan
)

print("\nTEST 1 — Unauthorized tool")
print("Selected tool:", result_1["tool"])
print("Planner source:", result_1["planner_source"])

assert result_1["tool"] != "delete_database"
assert result_1["tool"] is None or (
    result_1["tool"] in AGENT_TOOL_FUNCTIONS
)

print("PASS: Unauthorized tool was not accepted")


# ------------------------------------------------------------
# TEST 2 — Invented product ID
# ------------------------------------------------------------

invented_id_plan = {
    "status": "parsed",
    "plan": {
        "tool": "get_product_reviews",
        "query": "",
        "positions": [2],
        "product_ids": ["amazon_FAKE_PRODUCT_123"],
        "reason": "Review request"
    },
    "raw_output": "{}"
}

result_2 = validate_llm_tool_plan(
    message="Cho tôi xem đánh giá của sản phẩm thứ hai",
    session_id=safety_session,
    planner_result=invented_id_plan
)

print("\nTEST 2 — Invented product ID")
print("Arguments:", result_2["arguments"])

assert result_2["tool"] == "get_product_reviews"

assert result_2["arguments"]["product_id"] == saved_ids[1]

assert (
    result_2["arguments"]["product_id"]
    != "amazon_FAKE_PRODUCT_123"
)

print("PASS: Product ID resolved from session")


# ------------------------------------------------------------
# TEST 3 — Invalid JSON
# ------------------------------------------------------------

invalid_json_plan = {
    "status": "invalid_json",
    "plan": None,
    "raw_output": "not valid json",
    "error": "JSON parsing failed"
}

result_3 = validate_llm_tool_plan(
    message="Tìm microSD 128GB dưới 25 USD",
    session_id=safety_session,
    planner_result=invalid_json_plan
)

print("\nTEST 3 — Invalid JSON")
print("Selected tool:", result_3["tool"])
print("Planner source:", result_3["planner_source"])

assert (
    result_3["planner_source"]
    == "rule_based_fallback"
)

assert result_3["tool"] == "search_products"

print("PASS: Invalid JSON handled by fallback")


print("\n" + "=" * 80)
print("PASS: TOOL PLANNER SAFETY TEST")

STEP 16 — TOOL PLANNER SAFETY TEST
Session products: ['amazon_B0BRZPCQNJ', 'amazon_B09HKJY2QB']

TEST 1 — Unauthorized tool
Selected tool: None
Planner source: rule_based_fallback
PASS: Unauthorized tool was not accepted

TEST 2 — Invented product ID
Arguments: {'product_id': 'amazon_B09HKJY2QB', 'query': 'Cho tôi xem đánh giá của sản phẩm thứ hai', 'max_reviews': 3}
PASS: Product ID resolved from session

TEST 3 — Invalid JSON
Selected tool: search_products
Planner source: rule_based_fallback
PASS: Invalid JSON handled by fallback

PASS: TOOL PLANNER SAFETY TEST


In [60]:
# ============================================================
# STEP 16 — CELL 41
# CROSS-SESSION REFERENCE TEST
# ============================================================

print("=" * 80)
print("STEP 16 — CROSS-SESSION REFERENCE TEST")
print("=" * 80)


# Tạo phiên mới, không có lịch sử tìm kiếm.
isolated_session = create_chat_session(
    user_id=None
)

print(
    "Initial product IDs:",
    get_chat_session(
        isolated_session
    )["last_product_ids"]
)

assert not get_chat_session(
    isolated_session
)["last_product_ids"]


# Không cần gọi Qwen để kiểm thử cơ chế
# xác thực tham chiếu sản phẩm.
simulated_plan = {
    "status": "parsed",
    "plan": {
        "tool": "get_product_reviews",
        "query": "",
        "positions": [2],
        "product_ids": [
            "amazon_B09HKJY2QB"
        ],
        "reason": "Review request"
    },
    "raw_output": "{}"
}

isolated_result = validate_llm_tool_plan(
    message="Cho tôi xem đánh giá của sản phẩm thứ hai",
    session_id=isolated_session,
    planner_result=simulated_plan
)

print("\nValidated plan:")
print(isolated_result)

assert isolated_result["tool"] is None

assert isolated_result[
    "needs_clarification"
] is True

print(
    "\nPASS: CROSS-SESSION REFERENCE ISOLATION"
)

STEP 16 — CROSS-SESSION REFERENCE TEST
Initial product IDs: []

Validated plan:
{'intent': 'get_product_reviews', 'tool': None, 'arguments': {}, 'needs_clarification': True, 'planner_source': 'qwen', 'planner_reason': 'Need exactly one unambiguous product reference'}

PASS: CROSS-SESSION REFERENCE ISOLATION


In [63]:
# ============================================================
# STEP 17 — CELL 42 (FIXED)
# E-COMMERCE AI AGENT — GRADIO WEB UI
# ============================================================

import gradio as gr
import inspect
import uuid
import traceback

print("=" * 80)
print("STEP 17 — GRADIO WEB UI (FIXED)")
print("=" * 80)

print("Gradio version:", gr.__version__)


# ============================================================
# 1. CREATE SESSION
# ============================================================

def new_web_session():
    """
    Tạo session hội thoại mới.

    Mỗi phiên trình duyệt có một session_id riêng.
    Không gán cố định KNOWN_USER cho khách truy cập.
    """

    return create_chat_session(
        user_id=None
    )


# ============================================================
# 2. CHAT RESPONSE HANDLER
# ============================================================

def respond_to_user(
    message,
    history,
    session_id
):

    history = history or []

    if not message or not message.strip():

        return (
            history,
            "",
            session_id,
            "Vui lòng nhập câu hỏi."
        )

    message = message.strip()

    if not session_id:

        session_id = new_web_session()

    try:

        # --------------------------------------------
        # Call complete LLM Agent
        # --------------------------------------------

        result = chat_with_llm_agent(
            message=message,
            session_id=session_id
        )

        answer = result.get(
            "response",
            "Không nhận được câu trả lời."
        )

        # --------------------------------------------
        # IMPORTANT:
        # Gradio messages format
        # --------------------------------------------

        updated_history = history + [
            {
                "role": "user",
                "content": message
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]

        # --------------------------------------------
        # Agent status
        # --------------------------------------------

        tool_name = (
            result.get("tool")
            or "Không sử dụng công cụ"
        )

        planner_source = result.get(
            "planner_source",
            "unknown"
        )

        response_mode = result.get(
            "response_mode",
            "unknown"
        )

        status = result.get(
            "status",
            "unknown"
        )

        debug_text = (
            f"Session: {session_id}\n"
            f"Status: {status}\n"
            f"Tool: {tool_name}\n"
            f"Planner: {planner_source}\n"
            f"Response mode: {response_mode}"
        )

        return (
            updated_history,
            "",
            session_id,
            debug_text
        )

    except Exception as exc:

        print("\n[WEB AGENT ERROR]")
        traceback.print_exc()

        error_response = (
            "Hệ thống gặp lỗi khi xử lý yêu cầu. "
            "Vui lòng thử lại."
        )

        updated_history = history + [
            {
                "role": "user",
                "content": message
            },
            {
                "role": "assistant",
                "content": error_response
            }
        ]

        debug_text = (
            f"Session: {session_id}\n"
            f"Error type: {type(exc).__name__}\n"
            f"Error: {str(exc)}"
        )

        return (
            updated_history,
            "",
            session_id,
            debug_text
        )


# ============================================================
# 3. RESET CHAT
# ============================================================

def reset_web_chat():

    new_session_id = new_web_session()

    return (
        [],
        "",
        new_session_id,
        "Đã tạo phiên hội thoại mới."
    )


# ============================================================
# 4. CUSTOM CSS
# ============================================================

CUSTOM_CSS = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
}

#app-title {
    text-align: center;
    padding: 16px;
}

#app-title h1 {
    font-size: 30px;
    font-weight: 750;
}

#chat-area {
    min-height: 520px;
}

#status-box textarea {
    font-family: monospace;
    font-size: 12px;
}
"""


# ============================================================
# 5. CHATBOT CONFIGURATION
# ============================================================

# Different Gradio versions support different
# Chatbot constructor arguments.

chatbot_signature = inspect.signature(
    gr.Chatbot
)

chatbot_kwargs = {
    "label": "Trợ lý mua sắm",
    "height": 520,
    "elem_id": "chat-area"
}

if "type" in chatbot_signature.parameters:

    # THIS FIXES THE TUPLES FORMAT ERROR.
    chatbot_kwargs["type"] = "messages"

else:

    raise RuntimeError(
        "Phiên bản Gradio hiện tại không hỗ trợ "
        "gr.Chatbot(type='messages'). "
        "Hãy cập nhật Gradio và khởi động lại "
        "Kaggle session trước khi chạy lại."
    )


# ============================================================
# 6. BUILD WEB UI
# ============================================================

with gr.Blocks(
    title="E-Commerce AI Agent",
    css=CUSTOM_CSS,
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="cyan"
    )
) as demo:

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    gr.HTML(
        """
        <div id="app-title">
            <h1>🛒 E-Commerce AI Agent</h1>

            <p>
                Qwen Tool Calling ·
                FAISS RAG ·
                BPR Recommendation ·
                Knowledge Editing
            </p>
        </div>
        """
    )

    # --------------------------------------------------------
    # Session state
    # --------------------------------------------------------

    session_state = gr.State(
        value=None
    )

    with gr.Row():

        # ====================================================
        # LEFT COLUMN — CHAT
        # ====================================================

        with gr.Column(scale=3):

            chatbot = gr.Chatbot(
                **chatbot_kwargs
            )

            message_box = gr.Textbox(
                label="Nhập câu hỏi",
                placeholder=(
                    "Ví dụ: Gợi ý microSD "
                    "64GB dưới 20 USD..."
                ),
                lines=2
            )

            with gr.Row():

                send_button = gr.Button(
                    "Gửi tin nhắn",
                    variant="primary"
                )

                clear_button = gr.Button(
                    "Hội thoại mới",
                    variant="secondary"
                )

        # ====================================================
        # RIGHT COLUMN — AGENT INFO
        # ====================================================

        with gr.Column(scale=1):

            gr.Markdown(
                "### Thông tin Agent"
            )

            status_box = gr.Textbox(
                label="Trạng thái thực thi",
                lines=8,
                interactive=False,
                elem_id="status-box"
            )

            gr.Markdown(
                """
                ### Công nghệ

                **Qwen 2.5 3B**

                Lập kế hoạch gọi công cụ và
                sinh câu trả lời.

                **FAISS**

                Tìm kiếm sản phẩm theo ngữ nghĩa.

                **BPR**

                Cá nhân hóa gợi ý dựa trên
                lịch sử tương tác.

                **RAG**

                Truy xuất thông tin sản phẩm
                và đánh giá khách hàng.

                **Knowledge Editing**

                Áp dụng các bản chỉnh sửa tri thức
                đang kích hoạt tại thời điểm truy xuất.
                """
            )

            gr.Markdown(
                """
                **Lưu ý:** Giá hiển thị là dữ liệu
                lịch sử. Hệ thống chưa xác minh
                giá hiện tại hoặc tồn kho.
                """
            )

    # ========================================================
    # 7. EVENTS
    # ========================================================

    send_button.click(
        fn=respond_to_user,
        inputs=[
            message_box,
            chatbot,
            session_state
        ],
        outputs=[
            chatbot,
            message_box,
            session_state,
            status_box
        ]
    )

    message_box.submit(
        fn=respond_to_user,
        inputs=[
            message_box,
            chatbot,
            session_state
        ],
        outputs=[
            chatbot,
            message_box,
            session_state,
            status_box
        ]
    )

    clear_button.click(
        fn=reset_web_chat,
        inputs=[],
        outputs=[
            chatbot,
            message_box,
            session_state,
            status_box
        ]
    )


print("=" * 80)
print("PASS: GRADIO CHATBOT UI DEFINED")
print("Chatbot format: messages")
print("=" * 80)

STEP 17 — GRADIO WEB UI (FIXED)
Gradio version: 5.50.0
PASS: GRADIO CHATBOT UI DEFINED
Chatbot format: messages


In [64]:
# ============================================================
# STEP 17 — CELL 43 (FIXED)
# LAUNCH GRADIO WEB UI
# ============================================================

print("=" * 80)
print("STEP 17 — LAUNCH GRADIO WEB UI")
print("=" * 80)

# Close old demo if it is still running.
try:
    demo.close()
except Exception:
    pass

demo.queue(
    default_concurrency_limit=1
)

demo.launch(
    share=True,
    debug=False,
    show_error=True
)

STEP 17 — LAUNCH GRADIO WEB UI
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://ff409d56c09c60b06d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
